In [1]:
import pandas as pd
import numpy as np

pr_df = pd.read_csv("converted_prs/prs.csv")
repo_df = pd.read_csv("converted_prs/repos.csv")
events_df = pd.read_csv("converted_prs/events.csv")
comments_df = pd.read_csv("converted_prs/comments.csv")

pr_df = pr_df.drop_duplicates("pr_id")
repo_df = repo_df.drop_duplicates()

In [2]:
pr_df[pr_df["pr_id"] == "PR_kwDODDsPzc6Gg64D"]

,pr_id,repo,pr_number,pr_title,pr_url,pr_body,pr_body_length,state,is_merged,is_draft,...,created_at,closed_at,merged_at,updated_at,code_additions,code_deletions,changed_files,base_branch_name,head_branch_name,auto_merge
95442,PR_kwDODDsPzc6Gg64D,trailheadapps/ebikes-manufacturing,249,build(deps): bump lwr from 0.12.4 to 0.16.1,https://github.com/trailheadapps/ebikes-manufa...,Bumps [lwr](https://github.com/salesforce-expe...,2258,CLOSED,False,False,...,2025-01-01T00:39:50Z,2025-01-15T13:51:55Z,NaN,2025-01-15T13:51:57Z,19644,10889,2,main,dependabot/npm_and_yarn/lwr-0.16.1,False


In [3]:
print(f"{len(pr_df)} prs in {pr_df['repo'].nunique()} repos")

206023 prs in 10259 repos


# Basic information

In [4]:
print(pr_df['is_merged'].value_counts())
print(pr_df['state'].value_counts())
print()
print(pr_df['is_merged'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')
print(pr_df['state'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

is_merged
True     105110
False    100913
Name: count, dtype: int64
state
MERGED    105110
CLOSED     87381
OPEN       13532
Name: count, dtype: int64

is_merged
True     51.0%
False    49.0%
Name: proportion, dtype: str
state
MERGED    51.0%
CLOSED    42.4%
OPEN       6.6%
Name: proportion, dtype: str


## Add data to closed events

In [5]:
# only keep definitive close event
closures = events_df[events_df["event_type"] == "ClosedEvent"].sort_values("created_at").groupby("pr_id", as_index=False).last()

# add dates data
closures["pr_created"] = closures["pr_id"].map(
    pr_df.drop_duplicates("pr_id")
         .set_index("pr_id")["created_at"]
)
closures["pr_created"] = pd.to_datetime(closures["pr_created"])
closures["created_at"] = pd.to_datetime(closures["created_at"])
closures["time_to_close"] = closures["created_at"] - closures["pr_created"]

# remove unused column in closeevents
closures.drop("data", axis=1, inplace=True)

closures

,pr_id,repo,event_nr,event_type,event_author,created_at,pr_created,time_to_close
0,PR_kwDN-EXOs5ssoA,jquery/jquery-mousewheel,6,ClosedEvent,timmywil,2025-11-18 18:54:34+00:00,2025-11-15 10:34:07+00:00,3 days 08:20:27
1,PR_kwDN-EXOtB-a_g,jquery/jquery-mousewheel,6,ClosedEvent,dependabot,2025-11-18 17:32:25+00:00,2025-11-18 14:04:58+00:00,0 days 03:27:27
2,PR_kwDOA-0yTc6YwXD2,RobLoach/libretro-dats,6,ClosedEvent,dependabot,2025-09-24 20:52:20+00:00,2025-06-03 06:48:26+00:00,113 days 14:03:54
3,PR_kwDOA-0yTc6haOSN,RobLoach/libretro-dats,5,ClosedEvent,RobLoach,2026-02-22 20:15:42+00:00,2025-07-30 18:03:53+00:00,207 days 02:11:49
4,PR_kwDOA-0yTc6qXo1P,RobLoach/libretro-dats,6,ClosedEvent,RobLoach,2026-02-22 20:15:42+00:00,2025-09-24 20:52:18+00:00,150 days 23:23:24
...,...,...,...,...,...,...,...,...
191870,PR_kwDOQxcslc67EilT,littlebearapps/outlook-assistant,6,ClosedEvent,nathanschram,2026-02-25 11:26:42+00:00,2025-12-31 06:38:42+00:00,56 days 04:48:00
191871,PR_kwDOQxcslc67Eixe,littlebearapps/outlook-assistant,6,ClosedEvent,dependabot,2026-02-09 13:52:38+00:00,2025-12-31 06:39:26+00:00,40 days 07:13:12
191872,PR_kwDOQxcslc67Fv3i,littlebearapps/outlook-assistant,4,ClosedEvent,nathanschram,2026-02-25 11:24:44+00:00,2025-12-31 10:28:46+00:00,56 days 00:55:58
191873,PR_kwDOQxcslc67Fv44,littlebearapps/outlook-assistant,4,ClosedEvent,nathanschram,2026-02-25 11:24:47+00:00,2025-12-31 10:28:51+00:00,56 days 00:55:56


In [6]:
pd.set_option('display.max_rows', None)
print(closures["event_author"].value_counts().head(100))
pd.set_option('display.max_rows', 10)

event_author
dependabot             75990
github-actions         17177
mergify                 1339
mtrezza                 1033
stefcameron             1019
ndittren                 842
lumirlumir               805
lambertf917              742
nidaqg                   734
arlac77                  729
sanidhyy                 720
mjeanroy                 705
DavidAnson               687
TobiTenno                643
hodgef                   616
adamlui                  582
david-luna               527
willdurand               523
tekton-robot             506
warmachine028            499
swissspidy               481
wellwelwel               472
owlstronaut              428
UlisesGascon             414
sebnitu                  381
cossssmin                374
alexander-akait          363
Kikobeats                360
tyrasd                   359
bcomnes                  357
FreeTubeBot              343
arteck                   327
tschaub                  314
smalluban                313
T

In [7]:
bots = { "dependabot", "github-actions", "mergify", "accept-to-ship-bot", "stale", "kodiakhq", "tekton-robot", "FreeTubeBot" }
closures["closed_by_bot"] = closures["event_author"].isin(bots)
closures

,pr_id,repo,event_nr,event_type,event_author,created_at,pr_created,time_to_close,closed_by_bot
0,PR_kwDN-EXOs5ssoA,jquery/jquery-mousewheel,6,ClosedEvent,timmywil,2025-11-18 18:54:34+00:00,2025-11-15 10:34:07+00:00,3 days 08:20:27,False
1,PR_kwDN-EXOtB-a_g,jquery/jquery-mousewheel,6,ClosedEvent,dependabot,2025-11-18 17:32:25+00:00,2025-11-18 14:04:58+00:00,0 days 03:27:27,True
2,PR_kwDOA-0yTc6YwXD2,RobLoach/libretro-dats,6,ClosedEvent,dependabot,2025-09-24 20:52:20+00:00,2025-06-03 06:48:26+00:00,113 days 14:03:54,True
3,PR_kwDOA-0yTc6haOSN,RobLoach/libretro-dats,5,ClosedEvent,RobLoach,2026-02-22 20:15:42+00:00,2025-07-30 18:03:53+00:00,207 days 02:11:49,False
4,PR_kwDOA-0yTc6qXo1P,RobLoach/libretro-dats,6,ClosedEvent,RobLoach,2026-02-22 20:15:42+00:00,2025-09-24 20:52:18+00:00,150 days 23:23:24,False
...,...,...,...,...,...,...,...,...,...
191870,PR_kwDOQxcslc67EilT,littlebearapps/outlook-assistant,6,ClosedEvent,nathanschram,2026-02-25 11:26:42+00:00,2025-12-31 06:38:42+00:00,56 days 04:48:00,False
191871,PR_kwDOQxcslc67Eixe,littlebearapps/outlook-assistant,6,ClosedEvent,dependabot,2026-02-09 13:52:38+00:00,2025-12-31 06:39:26+00:00,40 days 07:13:12,True
191872,PR_kwDOQxcslc67Fv3i,littlebearapps/outlook-assistant,4,ClosedEvent,nathanschram,2026-02-25 11:24:44+00:00,2025-12-31 10:28:46+00:00,56 days 00:55:58,False
191873,PR_kwDOQxcslc67Fv44,littlebearapps/outlook-assistant,4,ClosedEvent,nathanschram,2026-02-25 11:24:47+00:00,2025-12-31 10:28:51+00:00,56 days 00:55:56,False


In [8]:
dependabot_comments = comments_df[comments_df["comment_author"].eq("dependabot")].copy()
dependabot_comments["superseded"] = (
    dependabot_comments["comment_body"].fillna("")
    .str.contains("up-to-date now|Superseded by|updatable in another way", case=False, regex=True)
)
sup = dependabot_comments.groupby("pr_id", as_index=False)["superseded"].any()

In [9]:
pr_df = pr_df.merge(closures, on="pr_id", how="left")
pr_df = pr_df.merge(sup, on="pr_id", how="left")
pr_df

,pr_id,repo_x,pr_number,pr_title,pr_url,pr_body,pr_body_length,state,is_merged,is_draft,...,auto_merge,repo_y,event_nr,event_type,event_author,created_at_y,pr_created,time_to_close,closed_by_bot,superseded
0,PR_kwDODjJbPM6NAIQJ,UK-Export-Finance/dtfs2,4241,chore(deps): bump the docker-packages group ac...,https://github.com/UK-Export-Finance/dtfs2/pul...,Bumps the docker-packages group with 1 update ...,3225,CLOSED,False,False,...,False,NaN,NaN,NaN,NaN,NaT,NaT,NaT,NaN,False
1,PR_kwDOBmc5bs6NAIMT,WFCD/warframe-hub,1338,ci: bump cypress from 14.0.3 to 14.1.0,https://github.com/WFCD/warframe-hub/pull/1338,Bumps [cypress](https://github.com/cypress-io/...,5113,MERGED,True,False,...,False,WFCD/warframe-hub,10.0,ClosedEvent,TobiTenno,2025-03-01 02:01:05+00:00,2025-03-01 00:55:29+00:00,0 days 01:05:36,False,NaN
2,PR_kwDOBmc5bs6NAIFq,WFCD/warframe-hub,1337,ci: bump eslint-config-prettier from 10.0.1 to...,https://github.com/WFCD/warframe-hub/pull/1337,Bumps the eslint group with 1 update: [eslint-...,8477,MERGED,True,False,...,True,WFCD/warframe-hub,13.0,ClosedEvent,TobiTenno,2025-03-01 02:04:43+00:00,2025-03-01 00:55:08+00:00,0 days 01:09:35,False,NaN
3,PR_kwDOC8_tdM6NAF4b,fraxken/combine-async-iterators,114,chore(deps): bump the github-actions group wit...,https://github.com/fraxken/combine-async-itera...,Bumps the github-actions group with 4 updates:...,19007,MERGED,True,False,...,False,fraxken/combine-async-iterators,5.0,ClosedEvent,github-actions,2025-03-01 00:46:01+00:00,2025-03-01 00:45:13+00:00,0 days 00:00:48,True,NaN
4,PR_kwDOGtgBvM6NAFeh,flotiq/flotiq-gatsby-portfolio-1,119,Bump node from 23.8.0 to 23.9.0,https://github.com/flotiq/flotiq-gatsby-portfo...,Bumps [node](https://github.com/aredridel/node...,2187,CLOSED,False,False,...,False,flotiq/flotiq-gatsby-portfolio-1,5.0,ClosedEvent,dependabot,2025-03-03 08:43:52+00:00,2025-03-01 00:42:46+00:00,2 days 08:01:06,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206018,PR_kwDOI-PZhc6eM9HG,MrBoombastic/OSINTCord,45,Bump dotenv from 16.4.7 to 17.2.0,https://github.com/MrBoombastic/OSINTCord/pull/45,Bumps [dotenv](https://github.com/motdotla/dot...,6023,CLOSED,False,False,...,False,MrBoombastic/OSINTCord,7.0,ClosedEvent,dependabot,2025-07-24 23:53:33+00:00,2025-07-09 23:09:55+00:00,15 days 00:43:38,True,True
206019,PR_kwDOADS4Xs6eM7HA,swagger-api/swagger-client,4006,chore(deps-dev): bump webpack from 5.99.9 to 5...,https://github.com/swagger-api/swagger-client/...,Bumps [webpack](https://github.com/webpack/web...,6077,MERGED,True,False,...,False,swagger-api/swagger-client,11.0,ClosedEvent,dependabot,2025-07-09 23:10:25+00:00,2025-07-09 23:06:27+00:00,0 days 00:03:58,True,NaN
206020,PR_kwDOOspSC86eM6mc,SocketDev/socket-mcp,23,Bump zod from 3.25.76 to 4.0.0,https://github.com/SocketDev/socket-mcp/pull/23,Bumps [zod](https://github.com/colinhacks/zod)...,4391,CLOSED,False,False,...,False,SocketDev/socket-mcp,10.0,ClosedEvent,dependabot,2025-07-10 23:05:00+00:00,2025-07-09 23:05:18+00:00,0 days 23:59:42,True,True
206021,PR_kwDOOspSC86eM6gW,SocketDev/socket-mcp,22,Bump @anthropic-ai/dxt from 0.2.0 to 0.2.1,https://github.com/SocketDev/socket-mcp/pull/22,Bumps @anthropic-ai/dxt from 0.2.0 to 0.2.1.\n...,1995,MERGED,True,False,...,False,SocketDev/socket-mcp,8.0,ClosedEvent,bcomnes,2025-07-09 23:58:34+00:00,2025-07-09 23:05:13+00:00,0 days 00:53:21,False,NaN


In [10]:
def classify(row):
    if row["is_merged"]:
        return "merged"

    if row["state"] == "OPEN":
        return "open"

    if row["superseded"] == True:
        return "closed_superseded"
        
    if row["closed_by_bot"] == True:
        return "closed_bot"
        
    return "closed_human"

pr_df["outcome"] = pr_df.apply(classify, axis=1)
pr_df
    

,pr_id,repo_x,pr_number,pr_title,pr_url,pr_body,pr_body_length,state,is_merged,is_draft,...,repo_y,event_nr,event_type,event_author,created_at_y,pr_created,time_to_close,closed_by_bot,superseded,outcome
0,PR_kwDODjJbPM6NAIQJ,UK-Export-Finance/dtfs2,4241,chore(deps): bump the docker-packages group ac...,https://github.com/UK-Export-Finance/dtfs2/pul...,Bumps the docker-packages group with 1 update ...,3225,CLOSED,False,False,...,NaN,NaN,NaN,NaN,NaT,NaT,NaT,NaN,False,closed_human
1,PR_kwDOBmc5bs6NAIMT,WFCD/warframe-hub,1338,ci: bump cypress from 14.0.3 to 14.1.0,https://github.com/WFCD/warframe-hub/pull/1338,Bumps [cypress](https://github.com/cypress-io/...,5113,MERGED,True,False,...,WFCD/warframe-hub,10.0,ClosedEvent,TobiTenno,2025-03-01 02:01:05+00:00,2025-03-01 00:55:29+00:00,0 days 01:05:36,False,NaN,merged
2,PR_kwDOBmc5bs6NAIFq,WFCD/warframe-hub,1337,ci: bump eslint-config-prettier from 10.0.1 to...,https://github.com/WFCD/warframe-hub/pull/1337,Bumps the eslint group with 1 update: [eslint-...,8477,MERGED,True,False,...,WFCD/warframe-hub,13.0,ClosedEvent,TobiTenno,2025-03-01 02:04:43+00:00,2025-03-01 00:55:08+00:00,0 days 01:09:35,False,NaN,merged
3,PR_kwDOC8_tdM6NAF4b,fraxken/combine-async-iterators,114,chore(deps): bump the github-actions group wit...,https://github.com/fraxken/combine-async-itera...,Bumps the github-actions group with 4 updates:...,19007,MERGED,True,False,...,fraxken/combine-async-iterators,5.0,ClosedEvent,github-actions,2025-03-01 00:46:01+00:00,2025-03-01 00:45:13+00:00,0 days 00:00:48,True,NaN,merged
4,PR_kwDOGtgBvM6NAFeh,flotiq/flotiq-gatsby-portfolio-1,119,Bump node from 23.8.0 to 23.9.0,https://github.com/flotiq/flotiq-gatsby-portfo...,Bumps [node](https://github.com/aredridel/node...,2187,CLOSED,False,False,...,flotiq/flotiq-gatsby-portfolio-1,5.0,ClosedEvent,dependabot,2025-03-03 08:43:52+00:00,2025-03-01 00:42:46+00:00,2 days 08:01:06,True,True,closed_superseded
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206018,PR_kwDOI-PZhc6eM9HG,MrBoombastic/OSINTCord,45,Bump dotenv from 16.4.7 to 17.2.0,https://github.com/MrBoombastic/OSINTCord/pull/45,Bumps [dotenv](https://github.com/motdotla/dot...,6023,CLOSED,False,False,...,MrBoombastic/OSINTCord,7.0,ClosedEvent,dependabot,2025-07-24 23:53:33+00:00,2025-07-09 23:09:55+00:00,15 days 00:43:38,True,True,closed_superseded
206019,PR_kwDOADS4Xs6eM7HA,swagger-api/swagger-client,4006,chore(deps-dev): bump webpack from 5.99.9 to 5...,https://github.com/swagger-api/swagger-client/...,Bumps [webpack](https://github.com/webpack/web...,6077,MERGED,True,False,...,swagger-api/swagger-client,11.0,ClosedEvent,dependabot,2025-07-09 23:10:25+00:00,2025-07-09 23:06:27+00:00,0 days 00:03:58,True,NaN,merged
206020,PR_kwDOOspSC86eM6mc,SocketDev/socket-mcp,23,Bump zod from 3.25.76 to 4.0.0,https://github.com/SocketDev/socket-mcp/pull/23,Bumps [zod](https://github.com/colinhacks/zod)...,4391,CLOSED,False,False,...,SocketDev/socket-mcp,10.0,ClosedEvent,dependabot,2025-07-10 23:05:00+00:00,2025-07-09 23:05:18+00:00,0 days 23:59:42,True,True,closed_superseded
206021,PR_kwDOOspSC86eM6gW,SocketDev/socket-mcp,22,Bump @anthropic-ai/dxt from 0.2.0 to 0.2.1,https://github.com/SocketDev/socket-mcp/pull/22,Bumps @anthropic-ai/dxt from 0.2.0 to 0.2.1.\n...,1995,MERGED,True,False,...,SocketDev/socket-mcp,8.0,ClosedEvent,bcomnes,2025-07-09 23:58:34+00:00,2025-07-09 23:05:13+00:00,0 days 00:53:21,False,NaN,merged


In [11]:
outcome_counts = pr_df["outcome"].value_counts()
outcome_pct = pr_df["outcome"].value_counts(normalize=True) * 100

result = pd.DataFrame({
    "count": outcome_counts,
    "percent": outcome_pct.round(2)
})
print(result)

                    count  percent
outcome                           
merged             105110    51.02
closed_superseded   65842    31.96
closed_human        16780     8.14
open                13532     6.57
closed_bot           4759     2.31


In [12]:
# Time to close when superseded

pr_df[pr_df["superseded"] == True].describe()

,pr_number,pr_body_length,code_additions,code_deletions,changed_files,event_nr,time_to_close
count,65853.000000,65853.000000,65853.000000,65853.000000,65853.000000,65811.000000,65811
mean,1472.225032,11999.982598,459.567992,353.538761,1.880810,6.971357,25 days 02:00:20.215419
std,3703.421381,10256.180036,2483.092467,2087.678279,6.393751,2.884419,46 days 10:32:57.494634
min,1.000000,1609.000000,0.000000,0.000000,0.000000,2.000000,0 days 00:00:02
25%,200.000000,5781.000000,5.000000,5.000000,1.000000,5.000000,4 days 13:20:23.500000
50%,566.000000,9852.000000,26.000000,18.000000,2.000000,7.000000,8 days 00:33:20
75%,1326.000000,14629.000000,197.000000,136.000000,2.000000,8.000000,27 days 23:25:25.500000
max,73968.000000,66483.000000,281746.000000,270846.000000,1318.000000,49.000000,563 days 18:01:40


In [13]:
pr_df[pr_df["time_to_close"] == "454 days 22:54:54"]

,pr_id,repo_x,pr_number,pr_title,pr_url,pr_body,pr_body_length,state,is_merged,is_draft,...,repo_y,event_nr,event_type,event_author,created_at_y,pr_created,time_to_close,closed_by_bot,superseded,outcome
95648,PR_kwDOApnhXs6GhHPW,ioBroker/ioBroker.vis-timeandweather,59,Bump @iobroker/adapter-core from 2.6.12 to 3.2.3,https://github.com/ioBroker/ioBroker.vis-timea...,Bumps [@iobroker/adapter-core](https://github....,6946,CLOSED,False,False,...,ioBroker/ioBroker.vis-timeandweather,4.0,ClosedEvent,dependabot,2026-04-01 02:12:02+00:00,2025-01-01 03:17:08+00:00,454 days 22:54:54,True,True,closed_superseded


In [14]:
repo_df["last_commit"] = pd.to_datetime(repo_df["last_commit"])

repo_df = (
    repo_df.sort_values("last_commit", ascending=False)
           .drop_duplicates(subset="repo_name", keep="first")
)

In [15]:
pr_df = pr_df.merge(
    repo_df,
    left_on="repo_x",
    right_on="repo_name",
    how="left",
    suffixes=("", "_r")
)

In [16]:
pr_df

,pr_id,repo_x,pr_number,pr_title,pr_url,pr_body,pr_body_length,state,is_merged,is_draft,...,repo_name,stars,forks,watchers,diskusage,primary_language,created_at,last_commit,issue_count,pr_count
0,PR_kwDODjJbPM6NAIQJ,UK-Export-Finance/dtfs2,4241,chore(deps): bump the docker-packages group ac...,https://github.com/UK-Export-Finance/dtfs2/pul...,Bumps the docker-packages group with 1 update ...,3225,CLOSED,False,False,...,UK-Export-Finance/dtfs2,19,7,2,144729,JavaScript,2020-02-04T10:37:22Z,2026-08-24 08:18:34+00:00,10,24
1,PR_kwDOBmc5bs6NAIMT,WFCD/warframe-hub,1338,ci: bump cypress from 14.0.3 to 14.1.0,https://github.com/WFCD/warframe-hub/pull/1338,Bumps [cypress](https://github.com/cypress-io/...,5113,MERGED,True,False,...,WFCD/warframe-hub,98,30,4,143793,TypeScript,2017-10-18T15:36:09Z,2026-08-29 01:45:44+00:00,1,0
2,PR_kwDOBmc5bs6NAIFq,WFCD/warframe-hub,1337,ci: bump eslint-config-prettier from 10.0.1 to...,https://github.com/WFCD/warframe-hub/pull/1337,Bumps the eslint group with 1 update: [eslint-...,8477,MERGED,True,False,...,WFCD/warframe-hub,98,30,4,143793,TypeScript,2017-10-18T15:36:09Z,2026-08-29 01:45:44+00:00,1,0
3,PR_kwDOC8_tdM6NAF4b,fraxken/combine-async-iterators,114,chore(deps): bump the github-actions group wit...,https://github.com/fraxken/combine-async-itera...,Bumps the github-actions group with 4 updates:...,19007,MERGED,True,False,...,fraxken/combine-async-iterators,18,5,0,497,JavaScript,2019-07-22T08:02:42Z,2026-08-14 00:43:11+00:00,0,0
4,PR_kwDOGtgBvM6NAFeh,flotiq/flotiq-gatsby-portfolio-1,119,Bump node from 23.8.0 to 23.9.0,https://github.com/flotiq/flotiq-gatsby-portfo...,Bumps [node](https://github.com/aredridel/node...,2187,CLOSED,False,False,...,flotiq/flotiq-gatsby-portfolio-1,17,4,4,10202,JavaScript,2022-01-21T05:22:50Z,2025-03-03 08:42:23+00:00,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206018,PR_kwDOI-PZhc6eM9HG,MrBoombastic/OSINTCord,45,Bump dotenv from 16.4.7 to 17.2.0,https://github.com/MrBoombastic/OSINTCord/pull/45,Bumps [dotenv](https://github.com/motdotla/dot...,6023,CLOSED,False,False,...,MrBoombastic/OSINTCord,13,1,1,282,JavaScript,2023-02-15T15:12:50Z,2025-02-23 22:30:36+00:00,0,7
206019,PR_kwDOADS4Xs6eM7HA,swagger-api/swagger-client,4006,chore(deps-dev): bump webpack from 5.99.9 to 5...,https://github.com/swagger-api/swagger-client/...,Bumps [webpack](https://github.com/webpack/web...,6077,MERGED,True,False,...,swagger-api/swagger-client,2694,765,83,60840,JavaScript,2012-02-16T00:08:43Z,2026-08-18 12:25:30+00:00,57,18
206020,PR_kwDOOspSC86eM6mc,SocketDev/socket-mcp,23,Bump zod from 3.25.76 to 4.0.0,https://github.com/SocketDev/socket-mcp/pull/23,Bumps [zod](https://github.com/colinhacks/zod)...,4391,CLOSED,False,False,...,SocketDev/socket-mcp,132,44,6,17262,TypeScript,2025-05-19T13:09:50Z,2026-08-27 22:04:54+00:00,3,0
206021,PR_kwDOOspSC86eM6gW,SocketDev/socket-mcp,22,Bump @anthropic-ai/dxt from 0.2.0 to 0.2.1,https://github.com/SocketDev/socket-mcp/pull/22,Bumps @anthropic-ai/dxt from 0.2.0 to 0.2.1.\n...,1995,MERGED,True,False,...,SocketDev/socket-mcp,132,44,6,17262,TypeScript,2025-05-19T13:09:50Z,2026-08-27 22:04:54+00:00,3,0


In [17]:
pr_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 206023 entries, 0 to 206022
Data columns (total 42 columns):
 #   Column              Non-Null Count   Dtype              
---  ------              --------------   -----              
 0   pr_id               206023 non-null  str                
 1   repo_x              206023 non-null  str                
 2   pr_number           206023 non-null  int64              
 3   pr_title            206023 non-null  str                
 4   pr_url              206023 non-null  str                
 5   pr_body             206022 non-null  str                
 6   pr_body_length      206023 non-null  int64              
 7   state               206023 non-null  str                
 8   is_merged           206023 non-null  bool               
 9   is_draft            206023 non-null  bool               
 10  mergeable           206023 non-null  str                
 11  merge_state_status  206023 non-null  str                
 12  created_at_x        206023 

In [18]:
pr_df["still_active"] = pr_df["last_commit"] > pd.to_datetime(pr_df["created_at_x"])

In [19]:
pr_df.value_counts(["still_active"])

still_active
True            192244
False            13779
Name: count, dtype: int64

In [20]:
# How many of the Open PR's are still_active

pr_df.groupby("outcome")["still_active"].value_counts()

outcome            still_active
closed_bot         True              4401
                   False              358
closed_human       True             16399
                   False              381
closed_superseded  True             59286
                   False             6556
merged             True            105095
                   False               15
open               True              7063
                   False             6469
Name: count, dtype: int64

The pr's that have been merged but still_active is False are results of using multiple branches. Still-active only checks the last commit made in the main branch. While a dependabot commit could have been pushed to another branch.

In [21]:
# Time to merge
pr_df["created"] = pd.to_datetime(pr_df["created_at_x"])
pr_df["ttm_h"] = (pd.to_datetime(pr_df["merged_at"]) - pr_df["created"]).dt.total_seconds() / 3600
pr_df["ttc_h"] = (pd.to_datetime(pr_df["closed_at"]) - pr_df["created"]).dt.total_seconds() / 3600
m = pr_df[pr_df.outcome == "merged"]
print("median:", m.ttm_h.median(), " < 1h | 24h | 7d:", (m.ttm_h <= 1).mean(), (m.ttm_h <= 24).mean(), (m.ttm_h <= 168).mean())
print("auto-merge share:", m.auto_merge.mean())
print(pr_df.groupby("outcome")["ttc_h"].median() / 24)   # median days to close
print("open in inactive repos:", (~pr_df[pr_df.outcome == "open"].still_active).sum())

median: 6.235694444444444  < 1h | 24h | 7d: 0.3381314813052992 0.6859004852059747 0.860374845400057
auto-merge share: 0.09051469888688041
outcome
closed_bot           4.514988
closed_human         3.394352
closed_superseded    8.026777
merged               0.259821
open                      NaN
Name: ttc_h, dtype: float64
open in inactive repos: 6469


In [22]:
# Check if body contains CVE or GSHA 
import re

keys = re.compile(r"GHSA-[\w-]{4,}|CVE-\d{4}-\d{4,}", re.IGNORECASE)
body = pr_df["pr_body"].fillna("")
pr_df["security_in_body"] = body.str.contains(keys)

In [23]:
print(pr_df["security_in_body"].value_counts())
print(pr_df["security_in_body"].value_counts(normalize=True).mul(100).round(1))

security_in_body
False    192775
True      13248
Name: count, dtype: int64
security_in_body
False    93.6
True      6.4
Name: proportion, dtype: float64


In [24]:
pd.crosstab(pr_df["security_in_body"], pr_df["outcome"], normalize="index")

outcome,closed_bot,closed_human,closed_superseded,merged,open
security_in_body,,,,,
False,0.022669,0.078874,0.324041,0.515145,0.059271
True,0.029363,0.118886,0.254755,0.438028,0.158967


## Packages

In [25]:
NAME = r"(?:`(?P<n1>[^`\n]+)`|\[(?P<n2>[^\]\n]+)\]\([^)\n]*\)|(?P<n3>[^\s`\[\n]+))"
VER  = r"v?\d[\w.\-+]*"
LEAD = r"(?:^|\band\s+)(?:[Uu]pdates?|[Bb]umps?)\s+"
END  = r"(?=[\s.,)]|$)"

ANCESTOR = re.compile(
    rf"{LEAD}[Aa]ncestor\s+dependency\s+{NAME}"
    rf"(?:\s+from\s+(?P<vfrom>{VER}))?"
    rf"(?:\s+to\s+(?P<vto>{VER}){END})?",
    flags=re.MULTILINE,
)
DIRECT = re.compile(
    rf"{LEAD}{NAME}"
    rf"(?:\s+from\s+(?P<vfrom>{VER}))?"
    rf"\s+to\s+(?P<vto>{VER}){END}",
    flags=re.MULTILINE,
)

FOOTER = re.compile(
    r"(?:You can trigger a rebase|Dependabot will resolve any conflicts|"
    r"<details>\s*<summary>Dependabot commands)",
    flags=re.IGNORECASE,
)
NAME_OK = re.compile(r"^(?:@[a-z0-9][\w.-]*/)?[a-z0-9][\w.-]*$", re.IGNORECASE)

def strip_footer(body):
    if not isinstance(body, str):
        return ""
    m = FOOTER.search(body)
    return body[:m.start()] if m else body

def parse_body(body):
    text = strip_footer(body)
    if not text:
        return []
    hits = []
    for pat, is_anc in ((ANCESTOR, True), (DIRECT, False)):
        for m in pat.finditer(text):
            hits.append((m.start(), m.end(), m, is_anc))
    hits.sort(key=lambda h: h[0])

    out, consumed = [], []
    for start, end, m, is_anc in hits:
        if any(start < e and end > s for s, e in consumed):   # ancestor wins overlaps
            continue
        consumed.append((start, end))
        name = m.group("n1") or m.group("n2") or m.group("n3")
        vto = m.group("vto")
        out.append((name, m.group("vfrom"), vto.rstrip(".") if vto else None, is_anc))
    return out

def classify_multi(title):
    if isinstance(title, str) and re.search(r"bump\s+the\s+.+?\s+group", title, re.IGNORECASE):
        return "group-bump"
    return "multi-bump"

# 1. single bump: "Bump X from A to B"
title_df = pr_df["pr_title"].str.extract(
    r"Bump\s+`?(.+?)`?(?:\s+from\s+(.+?))?\s+to\s+(\S+)",
    flags=re.IGNORECASE,
)
title_df.columns = ["package_name", "version_from", "version_to"]
title_df = title_df.astype(object)
title_df["is_ancestor"] = pd.NA
title_df[["pr_body", "pr_original_title"]] = pr_df[["pr_body", "pr_title"]]
title_df["type"] = pd.NA
title_df.loc[title_df["package_name"].notna(), "type"] = "bump"

# 2. "Update X requirement from A to B [in /dir]"
mask = title_df["package_name"].isna()
req = title_df.loc[mask, "pr_original_title"].str.extract(
    r"Update (\S+) requirement from (.+?)(?: to (.+))?$",
    flags=re.IGNORECASE,
)
req[2] = req[2].str.replace(r"\s+in\s+/.*$", "", regex=True)
title_df.loc[mask, "package_name"] = req[0]
title_df.loc[mask, "version_from"] = req[1]
title_df.loc[mask, "version_to"]   = req[2]
title_df.loc[mask & title_df["package_name"].notna(), "type"] = "requirement"

# 3. multiple packages, parsed from the body
mask = title_df["package_name"].isna()
matches = pr_df.loc[mask, "pr_body"].apply(parse_body)
has_body_matches = matches.str.len() > 0

title_df.loc[mask, "package_name"] = matches.apply(lambda m: [t[0] for t in m] or pd.NA)
title_df.loc[mask, "version_from"] = matches.apply(lambda m: [t[1] for t in m] or pd.NA)
title_df.loc[mask, "version_to"]   = matches.apply(lambda m: [t[2] for t in m] or pd.NA)
title_df.loc[mask, "is_ancestor"]  = matches.apply(lambda m: [t[3] for t in m] or pd.NA)

filled = mask & has_body_matches
title_df.loc[filled, "type"] = title_df.loc[filled, "pr_original_title"].apply(classify_multi)

# 4. title fallback: "Bump X and Y" with no usable body
mask2 = mask & ~has_body_matches
multi = pr_df.loc[mask2, "pr_title"].str.extract(
    r"[Bb]ump (.+?) and (\S+)", flags=re.IGNORECASE
)
title_df.loc[mask2, "package_name"] = multi.apply(
    lambda r: [r[0], r[1]] if r.notna().all() else pd.NA, axis=1
)
title_df.loc[mask2 & title_df["package_name"].notna(), "type"] = "multi-bump"

# 5. last resort: "Bump the X group across N directories with N updates"
mask3 = mask2 & title_df["package_name"].isna()
group = pr_df.loc[mask3, "pr_title"].str.extract(
    r"[Bb]ump the (.+?) group", flags=re.IGNORECASE
)
title_df.loc[mask3, "package_name"] = group[0]
title_df.loc[mask3 & title_df["package_name"].notna(), "type"] = "group-bump"

In [26]:
title_df[title_df["version_from"].isna()]

,package_name,version_from,version_to,is_ancestor,pr_body,pr_original_title,type
871,"[chai, @types/chai]",<NA>,<NA>,<NA>,Dependabot will resolve any conflicts with thi...,bump chai and @types/chai,multi-bump
928,"[chai, @types/chai]",<NA>,<NA>,<NA>,You can trigger a rebase of this PR by comment...,bump chai and @types/chai,multi-bump
951,"[chai, @types/chai]",<NA>,<NA>,<NA>,You can trigger a rebase of this PR by comment...,bump chai and @types/chai,multi-bump
1354,python-packages,<NA>,<NA>,<NA>,Dependabot will resolve any conflicts with thi...,Bump the python-packages group with 3 updates,group-bump
1450,python-packages,<NA>,<NA>,<NA>,\n\nDependabot will resolve any conflicts with...,Bump the python-packages group across 1 direct...,group-bump
...,...,...,...,...,...,...,...
197543,npm-development,<NA>,<NA>,<NA>,Dependabot will resolve any conflicts with thi...,chore(deps-dev): bump the npm-development grou...,group-bump
202103,NaN,<NA>,<NA>,<NA>,Updates the requirements on [actions/checkout]...,⬆️👷 Update actions/checkout requirement to 71c...,<NA>
204829,"[tmp, eslint]",<NA>,<NA>,<NA>,Dependabot will resolve any conflicts with thi...,build(deps): bump tmp and eslint,multi-bump
204836,"[tmp, @storybook/react, umi]",<NA>,<NA>,<NA>,\n\nDependabot will resolve any conflicts with...,"chore(deps): bump tmp, @storybook/react and umi",multi-bump


In [27]:
title_df = title_df.dropna(subset=["type"])

pr_df["package_name"]         = title_df["package_name"].reindex(pr_df.index)
pr_df["package_version_from"] = title_df["version_from"].reindex(pr_df.index)
pr_df["package_version_to"]   = title_df["version_to"].reindex(pr_df.index)

In [28]:
pr_df.at[898, "pr_url"]

'https://github.com/meilisearch/mini-dashboard/pull/597'

In [29]:
# Format of dependabot/(ecoystem)/(package) And dependabot-(ecosystem)-(package)
pr_df["package_ecosystem"] = pr_df["head_branch_name"].str.extract(r"dependabot(?:/|-)([^/-]+(?:_[^/-]+)*)")

ecosystem_mapping = {
    "npm_and_yarn": "NPM",
    "pip": "PIP",
    "composer": "COMPOSER",
    "docker": "DOCKER",
    "github_actions": "ACTIONS", 
    "go_modules": "GO",
    "maven": "MAVEN",
    "cargo": "RUST",
    "nuget": "NUGET",
    "gradle": "MAVEN",
    "submodules": "SM",
    "bundler": "RUBYGEMS",
    "hex": "ERLANG",
    "terraform": "TERRAFORM",
    "bun": "NPM",
    "pub": "PUB",
    "conda": "CONDA",
    "uv": "PIP",
    "helm": "HELM",
    "docker_compose": "DOCKER",
    "swift": "SWIFT",
}


print(pr_df["package_ecosystem"].value_counts()[:15])

pr_df["package_ecosystem"] = (
    pr_df["package_ecosystem"]
    .map(ecosystem_mapping)
)


package_ecosystem
npm_and_yarn      175954
github_actions     15350
pip                 4360
maven               2439
bundler             2143
                   ...  
gradle               379
submodules           196
bun                  160
hex                  135
uv                    94
Name: count, Length: 15, dtype: int64


In [30]:
pr_df[pr_df["package_name"].isna()]

,pr_id,repo_x,pr_number,pr_title,pr_url,pr_body,pr_body_length,state,is_merged,is_draft,...,pr_count,still_active,created,ttm_h,ttc_h,security_in_body,package_name,package_version_from,package_version_to,package_ecosystem
44572,PR_kwDODSNTp86jczRQ,amannn/action-semantic-pull-request,287,feat!: Upgrade action to use Node.js 24 and ESM,https://github.com/amannn/action-semantic-pull...,… as well as some internal changes:\r\n- Upgra...,241,MERGED,True,False,...,4,True,2025-08-13 10:04:39+00:00,2.916389,2.916389,False,NaN,NaN,NaN,NPM
53756,PR_kwDOBg0oS86g9tuY,percy/percy-storybook,1164,⬆️👷 Update actions/checkout requirement to 8ed...,https://github.com/percy/percy-storybook/pull/...,Updates the requirements on [actions/checkout]...,8321,CLOSED,False,False,...,17,True,2025-07-28 12:06:30+00:00,NaN,675.790833,False,NaN,NaN,NaN,ACTIONS
58421,PR_kwDOBg0oS861IQa7,percy/percy-storybook,1230,⬆️👷 Update actions/checkout requirement to c2d...,https://github.com/percy/percy-storybook/pull/...,Updates the requirements on [actions/checkout]...,8201,CLOSED,False,False,...,17,True,2025-11-24 08:02:43+00:00,NaN,335.991944,False,NaN,NaN,NaN,ACTIONS
67528,PR_kwDOJMdYcc6ffBhU,skills/secure-code-game,153,Bump brace-expansion in /Season-2/Level-3,https://github.com/skills/secure-code-game/pul...,\n\nDependabot will resolve any conflicts with...,1759,MERGED,True,False,...,2,True,2025-07-18 01:30:04+00:00,7.985278,7.985278,False,NaN,NaN,NaN,NPM
78162,PR_kwDOLGFzGc6WwqTU,microsoft/ABS-CSA-Assets,50,Update System.Text.Json to 6.0.10,https://github.com/microsoft/ABS-CSA-Assets/pu...,Performed the following updates:\n- Updated Sy...,2408,OPEN,False,False,...,4,True,2025-05-19 17:13:40+00:00,NaN,NaN,False,NaN,NaN,NaN,NUGET
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168859,PR_kwDOD59iSM6gD7NC,iobroker-community-adapters/ioBroker.alias-man...,132,Bump form-data,https://github.com/iobroker-community-adapters...,You can trigger a rebase of this PR by comment...,1800,CLOSED,False,False,...,5,True,2025-07-22 12:28:15+00:00,NaN,4919.896667,False,NaN,NaN,NaN,NPM
189025,PR_kwDOLYopoM6IjX9a,mrhenry/polyfill-library,96,Update dependencies,https://github.com/mrhenry/polyfill-library/pu...,NaN,0,MERGED,True,False,...,7,True,2025-01-21 22:33:40+00:00,2756.599167,2756.599444,False,NaN,NaN,NaN,NPM
192600,PR_kwDOBGDcPs6eAVT3,openhab/openhab-cloud,500,Bump semver,https://github.com/openhab/openhab-cloud/pull/500,\n\nDependabot will resolve any conflicts with...,1757,MERGED,True,False,...,27,True,2025-07-08 19:59:58+00:00,222.701944,222.701944,False,NaN,NaN,NaN,NPM
195418,PR_kwDOHUhT3s6mX4YN,gacts/gitleaks,86,⬆️ chore: Upgrade to Node 24 and update depend...,https://github.com/gacts/gitleaks/pull/86,Updates the Node.js version to 24 across the p...,198,MERGED,True,False,...,3,True,2025-09-02 05:10:26+00:00,1.937500,1.937500,False,NaN,NaN,NaN,NPM


In [31]:
pr_df["package_name"]

0         [node, node, node, node, node, nginx, node, node]
1                                                   cypress
2                                    eslint-config-prettier
3         [step-security/harden-runner, github/codeql-ac...
4                                                      node
                                ...                        
206018                                               dotenv
206019                                              webpack
206020                                                  zod
206021                                    @anthropic-ai/dxt
206022                                          @types/node
Name: package_name, Length: 206023, dtype: object

In [32]:
from itertools import zip_longest

def aslist(v):
    if isinstance(v, list):
        return v
    return [] if v is None or (np.ndim(v) == 0 and pd.isna(v)) else [v]

def clean(name):
    name = str(name).replace("\u200b", "").strip().strip("`")
    name = re.sub(r"</?code>", "", name)
    name = re.sub(r"^\[(.+?)\]\(.*\)$", r"\1", name)
    name = re.sub(r"^(?:the\s+)?dependency\s+", "", name, flags=re.I)
    return name.strip()

bumps = pd.DataFrame(
    [
        (pid, eco, clean(n), f, t)
        for pid, eco, ns, fs, ts in pr_df[
            ["pr_id", "package_ecosystem", "package_name",
             "package_version_from", "package_version_to"]
        ].itertuples(index=False)
        for n, f, t in zip_longest(aslist(ns), aslist(fs), aslist(ts), fillvalue=pd.NA)
        if pd.notna(n)
    ],
    columns=["pr_id", "ecosystem", "package_name", "version_from", "version_to"],
)
bumps[["ecosystem","package_name"]].drop_duplicates().to_csv("packages.csv", index=False)

In [33]:
pr_df

,pr_id,repo_x,pr_number,pr_title,pr_url,pr_body,pr_body_length,state,is_merged,is_draft,...,pr_count,still_active,created,ttm_h,ttc_h,security_in_body,package_name,package_version_from,package_version_to,package_ecosystem
0,PR_kwDODjJbPM6NAIQJ,UK-Export-Finance/dtfs2,4241,chore(deps): bump the docker-packages group ac...,https://github.com/UK-Export-Finance/dtfs2/pul...,Bumps the docker-packages group with 1 update ...,3225,CLOSED,False,False,...,24,True,2025-03-01 00:55:54+00:00,NaN,61.577500,False,"[node, node, node, node, node, nginx, node, node]","[23.4.0-alpine3.20, 23.4.0-alpine3.20, 23.4.0-...","[23.9.0-alpine3.20, 23.9.0-alpine3.20, 23.9.0-...",DOCKER
1,PR_kwDOBmc5bs6NAIMT,WFCD/warframe-hub,1338,ci: bump cypress from 14.0.3 to 14.1.0,https://github.com/WFCD/warframe-hub/pull/1338,Bumps [cypress](https://github.com/cypress-io/...,5113,MERGED,True,False,...,0,True,2025-03-01 00:55:29+00:00,1.093056,1.093056,False,cypress,14.0.3,14.1.0,NPM
2,PR_kwDOBmc5bs6NAIFq,WFCD/warframe-hub,1337,ci: bump eslint-config-prettier from 10.0.1 to...,https://github.com/WFCD/warframe-hub/pull/1337,Bumps the eslint group with 1 update: [eslint-...,8477,MERGED,True,False,...,0,True,2025-03-01 00:55:08+00:00,1.159722,1.159722,False,eslint-config-prettier,10.0.1,10.0.2,NPM
3,PR_kwDOC8_tdM6NAF4b,fraxken/combine-async-iterators,114,chore(deps): bump the github-actions group wit...,https://github.com/fraxken/combine-async-itera...,Bumps the github-actions group with 4 updates:...,19007,MERGED,True,False,...,0,True,2025-03-01 00:45:13+00:00,0.013333,0.013333,False,"[step-security/harden-runner, github/codeql-ac...","[2.10.4, 3.28.8, 2.4.0, 4.6.0]","[2.11.0, 3.28.10, 2.4.1, 4.6.1]",ACTIONS
4,PR_kwDOGtgBvM6NAFeh,flotiq/flotiq-gatsby-portfolio-1,119,Bump node from 23.8.0 to 23.9.0,https://github.com/flotiq/flotiq-gatsby-portfo...,Bumps [node](https://github.com/aredridel/node...,2187,CLOSED,False,False,...,5,True,2025-03-01 00:42:46+00:00,NaN,56.018333,False,node,23.8.0,23.9.0,NPM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206018,PR_kwDOI-PZhc6eM9HG,MrBoombastic/OSINTCord,45,Bump dotenv from 16.4.7 to 17.2.0,https://github.com/MrBoombastic/OSINTCord/pull/45,Bumps [dotenv](https://github.com/motdotla/dot...,6023,CLOSED,False,False,...,7,False,2025-07-09 23:09:55+00:00,NaN,360.727222,False,dotenv,16.4.7,17.2.0,NPM
206019,PR_kwDOADS4Xs6eM7HA,swagger-api/swagger-client,4006,chore(deps-dev): bump webpack from 5.99.9 to 5...,https://github.com/swagger-api/swagger-client/...,Bumps [webpack](https://github.com/webpack/web...,6077,MERGED,True,False,...,18,True,2025-07-09 23:06:27+00:00,0.066111,0.066111,False,webpack,5.99.9,5.100.0,NPM
206020,PR_kwDOOspSC86eM6mc,SocketDev/socket-mcp,23,Bump zod from 3.25.76 to 4.0.0,https://github.com/SocketDev/socket-mcp/pull/23,Bumps [zod](https://github.com/colinhacks/zod)...,4391,CLOSED,False,False,...,0,True,2025-07-09 23:05:18+00:00,NaN,23.995000,False,zod,3.25.76,4.0.0,NPM
206021,PR_kwDOOspSC86eM6gW,SocketDev/socket-mcp,22,Bump @anthropic-ai/dxt from 0.2.0 to 0.2.1,https://github.com/SocketDev/socket-mcp/pull/22,Bumps @anthropic-ai/dxt from 0.2.0 to 0.2.1.\n...,1995,MERGED,True,False,...,0,True,2025-07-09 23:05:13+00:00,0.889167,0.889167,False,@anthropic-ai/dxt,0.2.0,0.2.1,NPM


In [34]:
package_df = pd.read_csv("converted_prs/packages.csv", header=None, names=["ecosystem", "package_name", "ghsaId", "severity", "vulnerable_version", "patched_version"])
package_df

,ecosystem,package_name,ghsaId,severity,vulnerable_version,patched_version
0,NPM,eslint-config-prettier,GHSA-f29h-pxvx-f335,HIGH,">= 10.1.6, <= 10.1.7",10.1.8
1,NPM,eslint-config-prettier,GHSA-f29h-pxvx-f335,HIGH,= 9.1.1,9.1.2
2,NPM,eslint-config-prettier,GHSA-f29h-pxvx-f335,HIGH,= 8.10.1,8.10.2
3,ACTIONS,step-security/harden-runner,GHSA-46g3-37rh-v698,MODERATE,<= 2.15.1,2.16.0
4,ACTIONS,step-security/harden-runner,GHSA-g699-3x6g-wm3g,MODERATE,<= 2.15.1,2.16.0
...,...,...,...,...,...,...
10125,NPM,basic-ftp,GHSA-rp42-5vxx-qpwr,HIGH,<= 5.2.2,5.3.0
10126,NPM,basic-ftp,GHSA-6v7q-wjvx-w8wg,HIGH,<= 5.2.1,5.2.2
10127,NPM,basic-ftp,GHSA-chqc-8p9q-pq6q,HIGH,= 5.2.0,5.2.1
10128,NPM,basic-ftp,GHSA-5rq4-664w-9x2c,CRITICAL,< 5.2.0,5.2.0


In [35]:
print(package_df["patched_version"].dropna().unique()[:100])

<StringArray>
[          '10.1.8',            '9.1.2',           '8.10.2',
           '2.16.0',           '2.14.2',           '2.12.0',
           '2.10.2',           '3.28.3',           '8.5.23',
           '8.5.18',           '8.5.12',           '8.5.10',
           '8.4.31',           '7.0.36',           '8.2.13',
           '8.2.10',           '9.26.0',           '4.18.2',
           '4.59.0',           '3.30.0',           '2.80.0',
           '2.79.2',           '3.29.5',           '4.22.4',
           '1.13.0',           '7.29.6',       '8.0.0-rc.6',
           '11.1.1',           '13.0.1',           '12.0.1',
           '14.0.0',           '4.59.1',         '3.114.17',
           '2.20.2',           '3.19.0',           '2.20.1',
            '1.1.5',           '8.21.0',           '7.5.11',
            '6.2.4',            '5.2.5',           '8.20.1',
           '8.17.1',           '7.5.10',            '6.2.3',
            '5.2.4',            '1.0.1',            '5.2.3',
          

In [36]:
from packaging.version import parse as V
def vkey(s):
    m = re.search(r"\d+(?:\.\d+)*", str(s))   # grab the numeric version core
    if not m:
        return None
    try:
        return V(m.group(0))
    except InvalidVersion:
        return None

def in_vuln_range(from_v, to_v, vuln_range, patched_v):
    fv, tv, pv = vkey(from_v), vkey(to_v), vkey(patched_v)
    if fv is None or tv is None or pv is None:
        return False
    ok = True
    for part in str(vuln_range).split(","):
        m = re.match(r"\s*(>=|<=|>|<|=)\s*(.+)", part)
        if not m:
            continue
        op, b = m.group(1), vkey(m.group(2))
        if b is None:
            return False
        ok &= {">=": fv>=b, "<=": fv<=b, ">": fv>b, "<": fv<b, "=": fv==b}[op]
    return ok and tv >= pv
unsupported_ecosystems = ["DOCKER", "TERRAFORM", "CONDA", "HELM", "SM"]


merged = (
    bumps[~bumps["ecosystem"].isin(unsupported_ecosystems)]
    .merge(package_df, on=["ecosystem", "package_name"], how="left")
)

merged["bump_match"] = merged.apply(
    lambda r: in_vuln_range(r["version_from"], r["version_to"],
                            r["vulnerable_version"], r["patched_version"])
    if pd.notna(r["vulnerable_version"]) else False,
    axis=1,
)
sec = merged.groupby("pr_id")["bump_match"].any()
pr_df["is_security_bump"] = pr_df["pr_id"].map(sec)  


In [37]:
pr_df["is_security_bump"].value_counts()

is_security_bump
False    152852
True      51983
Name: count, dtype: int64

In [38]:
pr_df[pr_df["is_security_bump"] == True]

,pr_id,repo_x,pr_number,pr_title,pr_url,pr_body,pr_body_length,state,is_merged,is_draft,...,still_active,created,ttm_h,ttc_h,security_in_body,package_name,package_version_from,package_version_to,package_ecosystem,is_security_bump
49,PR_kwDOEiRQgc6M_7Rs,Ashteki/ashteki,1630,Bump @sentry/browser from 6.19.7 to 9.3.0,https://github.com/Ashteki/ashteki/pull/1630,Bumps [@sentry/browser](https://github.com/get...,14050,CLOSED,False,False,...,True,2025-03-01 00:04:33+00:00,NaN,119.726667,False,@sentry/browser,6.19.7,9.3.0,NPM,True
51,PR_kwDOBYAjhM6NAWMD,fastify/fastify-nextjs,1042,chore(deps-dev): Bump next from 13.4.3 to 15.2.0,https://github.com/fastify/fastify-nextjs/pull...,Bumps [next](https://github.com/vercel/next.js...,11239,CLOSED,False,False,...,True,2025-03-01 01:57:18+00:00,NaN,743.974167,False,next,13.4.3,15.2.0,NPM,True
54,PR_kwDOApKnas6NAWAz,brokenbot/sftp-promises,37,Bump serialize-javascript and mocha,https://github.com/brokenbot/sftp-promises/pul...,Bumps [serialize-javascript](https://github.co...,20653,OPEN,False,False,...,False,2025-03-01 01:56:04+00:00,NaN,NaN,False,"[serialize-javascript, mocha, serialize-javasc...","[None, None, 6.0.0, 9.1.2]","[6.0.2, None, 6.0.2, 11.1.0]",NPM,True
55,PR_kwDOApKnas6NAWAT,brokenbot/sftp-promises,36,Bump braces from 3.0.2 to 3.0.3,https://github.com/brokenbot/sftp-promises/pul...,Bumps [braces](https://github.com/micromatch/b...,4370,OPEN,False,False,...,False,2025-03-01 01:56:00+00:00,NaN,NaN,False,braces,3.0.2,3.0.3,NPM,True
109,PR_kwDOJA8JAc6NAkrY,bajrangCoder/acode-plugin-acodex,91,Bump esbuild from 0.19.12 to 0.25.0,https://github.com/bajrangCoder/acode-plugin-a...,Bumps [esbuild](https://github.com/evanw/esbui...,12929,MERGED,True,False,...,True,2025-03-01 02:57:38+00:00,0.013889,0.013889,True,esbuild,0.19.12,0.25.0,NPM,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
205970,PR_kwDOAVTE6c6eLHl3,skyvers/skyve,181,Bump commons-fileupload:commons-fileupload fro...,https://github.com/skyvers/skyve/pull/181,Bumps commons-fileupload:commons-fileupload fr...,2162,MERGED,True,False,...,True,2025-07-09 18:53:46+00:00,102.890833,102.890833,False,commons-fileupload:commons-fileupload,1.5,1.6.0,MAVEN,True
205971,PR_kwDOCn9EBc6eLHhs,rawdog71/Clownfish,125,Bump commons-fileupload:commons-fileupload fro...,https://github.com/rawdog71/Clownfish/pull/125,Bumps commons-fileupload:commons-fileupload fr...,2184,OPEN,False,False,...,True,2025-07-09 18:53:39+00:00,NaN,NaN,False,commons-fileupload:commons-fileupload,1.5,1.6.0,MAVEN,True
205984,PR_kwDOBngO-s6eLXEj,mistval/kotoba,406,Bump tar-fs from 2.1.2 to 2.1.3 in /api,https://github.com/mistval/kotoba/pull/406,Bumps [tar-fs](https://github.com/mafintosh/ta...,2617,MERGED,True,False,...,True,2025-07-09 19:24:27+00:00,0.054722,0.054722,False,tar-fs,2.1.2,2.1.3,NPM,True
205985,PR_kwDOBngO-s6eLXBs,mistval/kotoba,405,Bump brace-expansion from 1.1.11 to 1.1.12 in ...,https://github.com/mistval/kotoba/pull/405,Bumps [brace-expansion](https://github.com/jul...,3777,MERGED,True,False,...,True,2025-07-09 19:24:21+00:00,0.059167,0.059167,False,brace-expansion,1.1.11,1.1.12,NPM,True


In [39]:
#TODO: FIX OTHER TILES (MULTIPLE PACKAGES UPDATED AT ONCE)
pr_df[pr_df["package_name"].isna()]

,pr_id,repo_x,pr_number,pr_title,pr_url,pr_body,pr_body_length,state,is_merged,is_draft,...,still_active,created,ttm_h,ttc_h,security_in_body,package_name,package_version_from,package_version_to,package_ecosystem,is_security_bump
44572,PR_kwDODSNTp86jczRQ,amannn/action-semantic-pull-request,287,feat!: Upgrade action to use Node.js 24 and ESM,https://github.com/amannn/action-semantic-pull...,… as well as some internal changes:\r\n- Upgra...,241,MERGED,True,False,...,True,2025-08-13 10:04:39+00:00,2.916389,2.916389,False,NaN,NaN,NaN,NPM,NaN
53756,PR_kwDOBg0oS86g9tuY,percy/percy-storybook,1164,⬆️👷 Update actions/checkout requirement to 8ed...,https://github.com/percy/percy-storybook/pull/...,Updates the requirements on [actions/checkout]...,8321,CLOSED,False,False,...,True,2025-07-28 12:06:30+00:00,NaN,675.790833,False,NaN,NaN,NaN,ACTIONS,NaN
58421,PR_kwDOBg0oS861IQa7,percy/percy-storybook,1230,⬆️👷 Update actions/checkout requirement to c2d...,https://github.com/percy/percy-storybook/pull/...,Updates the requirements on [actions/checkout]...,8201,CLOSED,False,False,...,True,2025-11-24 08:02:43+00:00,NaN,335.991944,False,NaN,NaN,NaN,ACTIONS,NaN
67528,PR_kwDOJMdYcc6ffBhU,skills/secure-code-game,153,Bump brace-expansion in /Season-2/Level-3,https://github.com/skills/secure-code-game/pul...,\n\nDependabot will resolve any conflicts with...,1759,MERGED,True,False,...,True,2025-07-18 01:30:04+00:00,7.985278,7.985278,False,NaN,NaN,NaN,NPM,NaN
78162,PR_kwDOLGFzGc6WwqTU,microsoft/ABS-CSA-Assets,50,Update System.Text.Json to 6.0.10,https://github.com/microsoft/ABS-CSA-Assets/pu...,Performed the following updates:\n- Updated Sy...,2408,OPEN,False,False,...,True,2025-05-19 17:13:40+00:00,NaN,NaN,False,NaN,NaN,NaN,NUGET,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168859,PR_kwDOD59iSM6gD7NC,iobroker-community-adapters/ioBroker.alias-man...,132,Bump form-data,https://github.com/iobroker-community-adapters...,You can trigger a rebase of this PR by comment...,1800,CLOSED,False,False,...,True,2025-07-22 12:28:15+00:00,NaN,4919.896667,False,NaN,NaN,NaN,NPM,NaN
189025,PR_kwDOLYopoM6IjX9a,mrhenry/polyfill-library,96,Update dependencies,https://github.com/mrhenry/polyfill-library/pu...,NaN,0,MERGED,True,False,...,True,2025-01-21 22:33:40+00:00,2756.599167,2756.599444,False,NaN,NaN,NaN,NPM,NaN
192600,PR_kwDOBGDcPs6eAVT3,openhab/openhab-cloud,500,Bump semver,https://github.com/openhab/openhab-cloud/pull/500,\n\nDependabot will resolve any conflicts with...,1757,MERGED,True,False,...,True,2025-07-08 19:59:58+00:00,222.701944,222.701944,False,NaN,NaN,NaN,NPM,NaN
195418,PR_kwDOHUhT3s6mX4YN,gacts/gitleaks,86,⬆️ chore: Upgrade to Node 24 and update depend...,https://github.com/gacts/gitleaks/pull/86,Updates the Node.js version to 24 across the p...,198,MERGED,True,False,...,True,2025-09-02 05:10:26+00:00,1.937500,1.937500,False,NaN,NaN,NaN,NPM,NaN


## Finding direct reverts

In [40]:
pr_df[pr_df["security_in_body"] == True]

,pr_id,repo_x,pr_number,pr_title,pr_url,pr_body,pr_body_length,state,is_merged,is_draft,...,still_active,created,ttm_h,ttc_h,security_in_body,package_name,package_version_from,package_version_to,package_ecosystem,is_security_bump
109,PR_kwDOJA8JAc6NAkrY,bajrangCoder/acode-plugin-acodex,91,Bump esbuild from 0.19.12 to 0.25.0,https://github.com/bajrangCoder/acode-plugin-a...,Bumps [esbuild](https://github.com/evanw/esbui...,12929,MERGED,True,False,...,True,2025-03-01 02:57:38+00:00,0.013889,0.013889,True,esbuild,0.19.12,0.25.0,NPM,True
111,PR_kwDOB296yM6NAkVY,fastify/fastify-autoload,449,chore(deps-dev): bump esbuild from 0.24.2 to 0...,https://github.com/fastify/fastify-autoload/pu...,Bumps [esbuild](https://github.com/evanw/esbui...,9589,MERGED,True,False,...,True,2025-03-01 02:56:40+00:00,0.047222,0.047222,True,esbuild,0.24.2,0.25.0,NPM,True
112,PR_kwDOGecwK86NAj4T,jimmyl0l3c/cfg_share_links,330,Build(deps-dev): Bump vite from 5.4.11 to 6.2.0,https://github.com/jimmyl0l3c/cfg_share_links/...,Bumps [vite](https://github.com/vitejs/vite/tr...,16516,CLOSED,False,False,...,True,2025-03-01 02:55:00+00:00,NaN,167.862222,True,vite,5.4.11,6.2.0,NPM,True
122,PR_kwDOGBtTus6NAh-m,nextcloud/files_downloadlimit,473,chore(deps-dev): Bump vite from 5.4.9 to 6.2.0,https://github.com/nextcloud/files_downloadlim...,Bumps [vite](https://github.com/vitejs/vite/tr...,16514,CLOSED,False,False,...,True,2025-03-01 02:50:30+00:00,NaN,167.356944,True,vite,5.4.9,6.2.0,NPM,True
200,PR_kwDOLvSuxs6NAa66,nextcloud/whiteboard,386,chore(deps-dev): bump vite from 5.4.14 to 6.2.0,https://github.com/nextcloud/whiteboard/pull/386,Bumps [vite](https://github.com/vitejs/vite/tr...,16516,CLOSED,False,False,...,True,2025-03-01 02:19:26+00:00,NaN,169.084444,True,vite,5.4.14,6.2.0,NPM,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
205795,PR_kwDOB4HnKc6eDXiZ,josephfrazier/reported-web,593,build(deps): bump multer from 1.4.2 to 2.0.1,https://github.com/josephfrazier/reported-web/...,Bumps [multer](https://github.com/expressjs/mu...,17351,MERGED,True,False,...,True,2025-07-09 05:32:11+00:00,24.983333,24.983333,True,multer,1.4.2,2.0.1,NPM,True
205865,PR_kwDONbK8086eGWOP,ridwaanhall/rone-arena-api,34,chore(deps): bump urllib3 from 2.2.3 to 2.5.0,https://github.com/ridwaanhall/rone-arena-api/...,Bumps [urllib3](https://github.com/urllib3/url...,11885,MERGED,True,False,...,True,2025-07-09 10:36:45+00:00,0.126389,0.126389,True,urllib3,2.2.3,2.5.0,PIP,True
205891,PR_kwDOCzM2X86eGpm9,vpython/glowscript,198,Bump idna from 3.4 to 3.7,https://github.com/vpython/glowscript/pull/198,Bumps [idna](https://github.com/kjd/idna) from...,5873,MERGED,True,False,...,True,2025-07-09 11:06:03+00:00,25.596944,25.596944,True,idna,3.4,3.7,PIP,True
205894,PR_kwDOOFQTSM6eHkos,bugbasesecurity/pentest-copilot,11,Bump postcss and react-vnc in /frontend,https://github.com/bugbasesecurity/pentest-cop...,Bumps [postcss](https://github.com/postcss/pos...,14944,OPEN,False,False,...,True,2025-07-09 12:36:23+00:00,NaN,NaN,True,"[postcss, react-vnc, postcss, react-vnc]","[None, None, 7.0.39, 0.5.16]","[8.4.31, None, 8.4.31, 3.1.0]",NPM,True


In [41]:
pr_df["head_branch_name"].str.startswith("revert").value_counts()

head_branch_name
False    206023
Name: count, dtype: int64

In [42]:
events_df[
    events_df["event_type"] == "CrossReferencedEvent"
]

,pr_id,repo,event_nr,event_type,event_author,created_at,data
123,PR_kwDOAzhuQM6NAD2D,dotcore64/react-stay-scrolled,4,CrossReferencedEvent,NaN,2025-03-01T00:38:27Z,"{""source_number"": 675, ""source_title"": ""chore(..."
137,PR_kwDOAzhuQM6NADzq,dotcore64/react-stay-scrolled,4,CrossReferencedEvent,NaN,2025-03-01T00:38:13Z,"{""source_number"": 677, ""source_title"": ""chore(..."
172,PR_kwDOAzhuQM6NADuC,dotcore64/react-stay-scrolled,5,CrossReferencedEvent,NaN,2025-03-01T00:37:40Z,"{""source_number"": 661, ""source_title"": ""chore(..."
204,PR_kwDOEMp9fM6NACk9,terraform-linters/setup-tflint,3,CrossReferencedEvent,NaN,2025-03-01T00:33:27Z,"{""source_number"": 271, ""source_title"": ""build(..."
211,PR_kwDOEMp9fM6NAChU,terraform-linters/setup-tflint,3,CrossReferencedEvent,NaN,2025-03-01T00:33:08Z,"{""source_number"": 270, ""source_title"": ""build(..."
...,...,...,...,...,...,...,...
1921647,PR_kwDOA9Ajr86eMyQc,Zheaoli/zheaoli.github.io,5,CrossReferencedEvent,NaN,2025-07-09T22:44:30Z,"{""source_number"": 14, ""source_title"": ""Bump he..."
1921692,PR_kwDOJetNIM6eNL9p,neoxr/open-api,5,CrossReferencedEvent,NaN,2025-07-09T23:56:26Z,"{""source_number"": 21, ""source_title"": ""chore(d..."
1921704,PR_kwDOJ4IRwM6eNDhn,polly3d/mockai,8,CrossReferencedEvent,NaN,2025-07-09T23:27:18Z,"{""source_number"": 57, ""source_title"": ""chore(d..."
1921713,PR_kwDOGBHQFM6eNAjn,devinstewart/sns-payload-validator,5,CrossReferencedEvent,NaN,2025-07-09T23:16:53Z,"{""source_number"": 104, ""source_title"": ""Bump @..."


In [43]:
import json
import pandas as pd

crossdf = events_df[
    events_df["event_type"] == "CrossReferencedEvent"
].copy()

parsed_data = crossdf["data"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)

data_columns = pd.json_normalize(parsed_data)
data_columns.index = crossdf.index

result_df = pd.concat([crossdf, data_columns], axis=1)

result_df

,pr_id,repo,event_nr,event_type,event_author,created_at,data,source_number,source_title,source_state,source_merged,source_head_ref
123,PR_kwDOAzhuQM6NAD2D,dotcore64/react-stay-scrolled,4,CrossReferencedEvent,NaN,2025-03-01T00:38:27Z,"{""source_number"": 675, ""source_title"": ""chore(...",675.0,chore(deps-dev): bump eslint-plugin-react-hook...,CLOSED,False,dependabot/npm_and_yarn/eslint-plugin-react-ho...
137,PR_kwDOAzhuQM6NADzq,dotcore64/react-stay-scrolled,4,CrossReferencedEvent,NaN,2025-03-01T00:38:13Z,"{""source_number"": 677, ""source_title"": ""chore(...",677.0,chore(deps-dev): bump eslint from 8.57.0 to 9....,CLOSED,False,dependabot/npm_and_yarn/eslint-9.19.0
172,PR_kwDOAzhuQM6NADuC,dotcore64/react-stay-scrolled,5,CrossReferencedEvent,NaN,2025-03-01T00:37:40Z,"{""source_number"": 661, ""source_title"": ""chore(...",661.0,chore(deps-dev): bump eslint-plugin-unicorn fr...,CLOSED,False,dependabot/npm_and_yarn/eslint-plugin-unicorn-...
204,PR_kwDOEMp9fM6NACk9,terraform-linters/setup-tflint,3,CrossReferencedEvent,NaN,2025-03-01T00:33:27Z,"{""source_number"": 271, ""source_title"": ""build(...",271.0,build(deps-dev): Bump eslint-config-prettier f...,CLOSED,False,dependabot/npm_and_yarn/eslint-config-prettier...
211,PR_kwDOEMp9fM6NAChU,terraform-linters/setup-tflint,3,CrossReferencedEvent,NaN,2025-03-01T00:33:08Z,"{""source_number"": 270, ""source_title"": ""build(...",270.0,build(deps-dev): Bump @eslint/compat from 1.2....,CLOSED,False,dependabot/npm_and_yarn/eslint/compat-1.2.6
...,...,...,...,...,...,...,...,...,...,...,...,...
1921647,PR_kwDOA9Ajr86eMyQc,Zheaoli/zheaoli.github.io,5,CrossReferencedEvent,NaN,2025-07-09T22:44:30Z,"{""source_number"": 14, ""source_title"": ""Bump he...",14.0,Bump hexo-theme-butterfly from 5.3.5 to 5.4.1,CLOSED,False,dependabot/npm_and_yarn/hexo-theme-butterfly-5...
1921692,PR_kwDOJetNIM6eNL9p,neoxr/open-api,5,CrossReferencedEvent,NaN,2025-07-09T23:56:26Z,"{""source_number"": 21, ""source_title"": ""chore(d...",21.0,chore(deps): bump dotenv from 16.5.0 to 17.0.1,CLOSED,False,dependabot/npm_and_yarn/dotenv-17.0.1
1921704,PR_kwDOJ4IRwM6eNDhn,polly3d/mockai,8,CrossReferencedEvent,NaN,2025-07-09T23:27:18Z,"{""source_number"": 57, ""source_title"": ""chore(d...",57.0,chore(deps): bump dotenv from 16.4.7 to 17.1.0,CLOSED,False,dependabot/npm_and_yarn/dotenv-17.1.0
1921713,PR_kwDOGBHQFM6eNAjn,devinstewart/sns-payload-validator,5,CrossReferencedEvent,NaN,2025-07-09T23:16:53Z,"{""source_number"": 104, ""source_title"": ""Bump @...",104.0,Bump @types/node from 22.16.1 to 24.0.11,CLOSED,False,dependabot/npm_and_yarn/types/node-24.0.11


In [44]:
print(result_df["source_head_ref"].str.lower().str.startswith("revert").value_counts())
print(result_df["source_head_ref"].str.lower().str.startswith("revert").value_counts(normalize=True).mul(100).round(1))

source_head_ref
False    60816
True        97
Name: count, dtype: int64
source_head_ref
False    99.8
True      0.2
Name: proportion, dtype: float64


In [45]:
result_df[result_df["source_head_ref"].str.lower().str.startswith("revert") & result_df["source_merged"]]

,pr_id,repo,event_nr,event_type,event_author,created_at,data,source_number,source_title,source_state,source_merged,source_head_ref
4075,PR_kwDOI6VkUc6NArQ9,raschy/ioBroker.deyeidc,11,CrossReferencedEvent,NaN,2025-03-08T18:41:51Z,"{""source_number"": 174, ""source_title"": ""Revert...",174.0,"Revert ""Bump typescript from 5.7.3 to 5.8.2""",MERGED,True,revert-170-dependabot/npm_and_yarn/typescript-...
4128,PR_kwDOI6VkUc6NArPQ,raschy/ioBroker.deyeidc,11,CrossReferencedEvent,NaN,2025-03-08T18:33:06Z,"{""source_number"": 173, ""source_title"": ""Revert...",173.0,"Revert ""Bump chai and @types/chai""",MERGED,True,revert-169-dependabot/npm_and_yarn/multi-6156a...
6542,PR_kwDOBZpXw86NBGxL,iobroker-community-adapters/ioBroker.kecontact,11,CrossReferencedEvent,NaN,2025-03-01T20:40:43Z,"{""source_number"": 434, ""source_title"": ""Revert...",434.0,"Revert ""Bump typescript from 5.7.3 to 5.8.2""",MERGED,True,revert-431-dependabot/npm_and_yarn/typescript-...
12690,PR_kwDOCWQXL86NCfHS,meshery/meshery,11,CrossReferencedEvent,NaN,2025-03-07T20:29:47Z,"{""source_number"": 13927, ""source_title"": ""Reve...",13927.0,"Revert ""Bump @xstate/react from 4.1.3 to 5.0.2...",MERGED,True,revert-13818-dependabot/npm_and_yarn/ui/xstate...
43748,PR_kwDOAWBOZM644itv,axios/axios,13,CrossReferencedEvent,NaN,2026-01-19T19:37:10Z,"{""source_number"": 7334, ""source_title"": ""rever...",7334.0,revert(deps): bump peter-evans/create-pull-req...,MERGED,True,revert-7303-dependabot/github_actions/github-a...
...,...,...,...,...,...,...,...,...,...,...,...,...
1754483,PR_kwDOHFBhsc6MCnp8,ClickHouse/clickhouse-docs,9,CrossReferencedEvent,NaN,2025-02-24T09:31:19Z,"{""source_number"": 3334, ""source_title"": ""Rever...",3334.0,"Revert ""Bump esbuild-loader from 4.2.2 to 4.3.0""",MERGED,True,revert-3320-dependabot/npm_and_yarn/esbuild-lo...
1754493,PR_kwDOHFBhsc6MCm1w,ClickHouse/clickhouse-docs,9,CrossReferencedEvent,NaN,2025-02-24T13:44:23Z,"{""source_number"": 3337, ""source_title"": ""Rever...",3337.0,"Revert ""Bump actions/github-script from 6 to 7""",MERGED,True,revert-3318-dependabot/github_actions/actions/...
1816052,PR_kwDOEBB5W86TH0HI,josephdadams/TallyArbiter,6,CrossReferencedEvent,NaN,2025-04-18T14:06:36Z,"{""source_number"": 809, ""source_title"": ""Revert...",809.0,"Revert ""Bump esbuild and @angular-devkit/build...",MERGED,True,revert-807-dependabot/npm_and_yarn/UI/multi-2d...
1843003,PR_kwDOJDftgs6HrCb5,complexdatacollective/Fresco,7,CrossReferencedEvent,NaN,2025-01-14T08:41:43Z,"{""source_number"": 255, ""source_title"": ""Revert...",255.0,"Revert ""Bump next from 14.2.16 to 14.2.21 in t...",MERGED,True,revert-254-dependabot/npm_and_yarn/npm_and_yar...


In [46]:
reverted_pr_ids = result_df.loc[
    result_df["source_head_ref"].str.lower().str.startswith("revert", na=False)
    & result_df["source_merged"],
    "pr_id"
]

pr_df["reverted_ref"] = pr_df["pr_id"].isin(reverted_pr_ids)

In [47]:
# time to revert
rev = result_df[result_df.source_head_ref.str.lower().str.startswith("revert", na=False) & result_df.source_merged]
rt = pd.to_datetime(rev.groupby("pr_id")["created_at"].min())

lag = (rt - pd.to_datetime(pr_df.set_index("pr_id").loc[rt.index, "merged_at"])).dt.total_seconds() / 86400
print("median days:", lag.median(), " <= day:", (lag <= 1).sum(), " <= week:", (lag <= 7).sum(), " max:", lag.max())
print(pr_df[pr_df.reverted_ref].package_ecosystem.value_counts())

median days: 0.08774305555555556  <= day: 66  <= week: 77  max: 32.45068287037037
package_ecosystem
NPM         58
ACTIONS      6
MAVEN        5
PIP          4
RUBYGEMS     4
SM           2
GO           2
COMPOSER     1
Name: count, dtype: int64


In [48]:
pr_df

,pr_id,repo_x,pr_number,pr_title,pr_url,pr_body,pr_body_length,state,is_merged,is_draft,...,created,ttm_h,ttc_h,security_in_body,package_name,package_version_from,package_version_to,package_ecosystem,is_security_bump,reverted_ref
0,PR_kwDODjJbPM6NAIQJ,UK-Export-Finance/dtfs2,4241,chore(deps): bump the docker-packages group ac...,https://github.com/UK-Export-Finance/dtfs2/pul...,Bumps the docker-packages group with 1 update ...,3225,CLOSED,False,False,...,2025-03-01 00:55:54+00:00,NaN,61.577500,False,"[node, node, node, node, node, nginx, node, node]","[23.4.0-alpine3.20, 23.4.0-alpine3.20, 23.4.0-...","[23.9.0-alpine3.20, 23.9.0-alpine3.20, 23.9.0-...",DOCKER,NaN,False
1,PR_kwDOBmc5bs6NAIMT,WFCD/warframe-hub,1338,ci: bump cypress from 14.0.3 to 14.1.0,https://github.com/WFCD/warframe-hub/pull/1338,Bumps [cypress](https://github.com/cypress-io/...,5113,MERGED,True,False,...,2025-03-01 00:55:29+00:00,1.093056,1.093056,False,cypress,14.0.3,14.1.0,NPM,False,False
2,PR_kwDOBmc5bs6NAIFq,WFCD/warframe-hub,1337,ci: bump eslint-config-prettier from 10.0.1 to...,https://github.com/WFCD/warframe-hub/pull/1337,Bumps the eslint group with 1 update: [eslint-...,8477,MERGED,True,False,...,2025-03-01 00:55:08+00:00,1.159722,1.159722,False,eslint-config-prettier,10.0.1,10.0.2,NPM,False,False
3,PR_kwDOC8_tdM6NAF4b,fraxken/combine-async-iterators,114,chore(deps): bump the github-actions group wit...,https://github.com/fraxken/combine-async-itera...,Bumps the github-actions group with 4 updates:...,19007,MERGED,True,False,...,2025-03-01 00:45:13+00:00,0.013333,0.013333,False,"[step-security/harden-runner, github/codeql-ac...","[2.10.4, 3.28.8, 2.4.0, 4.6.0]","[2.11.0, 3.28.10, 2.4.1, 4.6.1]",ACTIONS,False,False
4,PR_kwDOGtgBvM6NAFeh,flotiq/flotiq-gatsby-portfolio-1,119,Bump node from 23.8.0 to 23.9.0,https://github.com/flotiq/flotiq-gatsby-portfo...,Bumps [node](https://github.com/aredridel/node...,2187,CLOSED,False,False,...,2025-03-01 00:42:46+00:00,NaN,56.018333,False,node,23.8.0,23.9.0,NPM,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206018,PR_kwDOI-PZhc6eM9HG,MrBoombastic/OSINTCord,45,Bump dotenv from 16.4.7 to 17.2.0,https://github.com/MrBoombastic/OSINTCord/pull/45,Bumps [dotenv](https://github.com/motdotla/dot...,6023,CLOSED,False,False,...,2025-07-09 23:09:55+00:00,NaN,360.727222,False,dotenv,16.4.7,17.2.0,NPM,False,False
206019,PR_kwDOADS4Xs6eM7HA,swagger-api/swagger-client,4006,chore(deps-dev): bump webpack from 5.99.9 to 5...,https://github.com/swagger-api/swagger-client/...,Bumps [webpack](https://github.com/webpack/web...,6077,MERGED,True,False,...,2025-07-09 23:06:27+00:00,0.066111,0.066111,False,webpack,5.99.9,5.100.0,NPM,False,False
206020,PR_kwDOOspSC86eM6mc,SocketDev/socket-mcp,23,Bump zod from 3.25.76 to 4.0.0,https://github.com/SocketDev/socket-mcp/pull/23,Bumps [zod](https://github.com/colinhacks/zod)...,4391,CLOSED,False,False,...,2025-07-09 23:05:18+00:00,NaN,23.995000,False,zod,3.25.76,4.0.0,NPM,False,False
206021,PR_kwDOOspSC86eM6gW,SocketDev/socket-mcp,22,Bump @anthropic-ai/dxt from 0.2.0 to 0.2.1,https://github.com/SocketDev/socket-mcp/pull/22,Bumps @anthropic-ai/dxt from 0.2.0 to 0.2.1.\n...,1995,MERGED,True,False,...,2025-07-09 23:05:13+00:00,0.889167,0.889167,False,@anthropic-ai/dxt,0.2.0,0.2.1,NPM,False,False


**Summary**:  
97 Total Crossreferences to reverts,  
82 of those are merged (so they are actually reverted) 
15 of those are closed (so not reverted)

### Repo Size

In [49]:
repos = repo_df.set_index("repo_name")[
    ["stars","forks","diskusage","primary_language"]]
repos

,stars,forks,diskusage,primary_language
repo_name,,,,
TA2k/ioBroker.bmw,29,11,2989,JavaScript
codeaashu/DevDisplay,330,468,281851,JavaScript
fontra/fontra,825,55,16542,JavaScript
arlac77/pratt-parser,17,4,3239,JavaScript
arlac77/svelte-session-manager,110,7,17393,JavaScript
...,...,...,...,...
ben-ng/minifyify,185,30,9203,JavaScript
puriketu99/porky,29,0,1083,JavaScript
hh54188/scrapy_douban,38,18,5608,JavaScript


In [50]:
print(len(repos), repos.index.nunique())        # must be equal — join breaks otherwise
print(repos["stars"].describe())

10259 10259
count     10259.000000
mean        867.363583
std        5628.844587
min           1.000000
25%          21.000000
50%          52.000000
75%         216.000000
max      244608.000000
Name: stars, dtype: float64


In [51]:
repos["bucket"] = pd.cut(repos["stars"], [0, 24, 99, 499, 4999, 10**9],
                      labels=["10–24","25–99","100–499","500–4999","5000+"])

In [52]:
repos["bucket"].value_counts()

bucket
25–99       3435
10–24       3059
100–499     2187
500–4999    1256
5000+        322
Name: count, dtype: int64

In [53]:
print(repos["bucket"].value_counts().sort_index())
print(repos["stars"].quantile([0, .25, .5, .75, .90, .95, .99, 1]))

bucket
10–24       3059
25–99       3435
100–499     2187
500–4999    1256
5000+        322
Name: count, dtype: int64
0.00         1.00
0.25        21.00
0.50        52.00
0.75       216.00
0.90       922.20
0.95      2598.10
0.99     17318.96
1.00    244608.00
Name: stars, dtype: float64


In [54]:
repos

,stars,forks,diskusage,primary_language,bucket
repo_name,,,,,
TA2k/ioBroker.bmw,29,11,2989,JavaScript,25–99
codeaashu/DevDisplay,330,468,281851,JavaScript,100–499
fontra/fontra,825,55,16542,JavaScript,500–4999
arlac77/pratt-parser,17,4,3239,JavaScript,10–24
arlac77/svelte-session-manager,110,7,17393,JavaScript,100–499
...,...,...,...,...,...
ben-ng/minifyify,185,30,9203,JavaScript,100–499
puriketu99/porky,29,0,1083,JavaScript,25–99
hh54188/scrapy_douban,38,18,5608,JavaScript,25–99


In [55]:
# Remove the old repository_size column if it exists
pr_df = pr_df.drop(columns=['repository_size'], errors='ignore')

# Create one row per repository
repo_sizes = (
    pr_df[['repo_x', 'stars']]
    .drop_duplicates(subset='repo_x')
    .copy()
)

# Calculate the median number of stars
stars_threshold = repo_sizes['stars'].median()

print(f"Number of unique repositories: {len(repo_sizes):,}")
print(f"Median stars threshold: {stars_threshold:,.0f}")

# Classify repositories
repo_sizes['repository_size'] = repo_sizes['stars'].apply(
    lambda x: 'Small' if x <= stars_threshold else 'Large'
)

# Check repository-level distribution
display(
    repo_sizes.groupby('repository_size')['stars']
    .agg(
        repositories='count',
        minimum='min',
        median='median',
        maximum='max'
    )
)

Number of unique repositories: 10,259
Median stars threshold: 52


,repositories,minimum,median,maximum
repository_size,,,,
Large,5094,53,218.0,244608
Small,5165,1,21.0,52


In [56]:
pr_df = pr_df.merge(
    repo_sizes[['repo_x', 'repository_size']],
    on='repo_x',
    how='left'
)

# Check PR counts
display(
    pr_df['repository_size']
    .value_counts()
)

repository_size
Large    113424
Small     92599
Name: count, dtype: int64

In [57]:
pr_df['is_rejected'] = pr_df['outcome'].isin(['closed_human'])
rq4_results = (
    pr_df
    .groupby('repository_size')
    .agg(
        total_prs=('repository_size', 'size'),
        merged=('is_merged', 'sum'),
        rejected=('is_rejected', 'sum'),
        reverted=('reverted_ref', 'sum'),
        superseded=('outcome', lambda s: (s == 'closed_superseded').sum()),
        open_prs=('outcome', lambda s: (s == 'open').sum()),
    )
)

for col in ['merged', 'rejected', 'superseded', 'open_prs']:
    rq4_results[f'{col}_rate'] = rq4_results[col] / rq4_results['total_prs'] * 100

# Revert rate among merged PRs
rq4_results['revert_rate'] = (
    rq4_results['reverted'] /
    rq4_results['merged'] * 100
)

rq4_results = rq4_results.round(2)

display(rq4_results)

,total_prs,merged,rejected,reverted,superseded,open_prs,merged_rate,rejected_rate,superseded_rate,open_prs_rate,revert_rate
repository_size,,,,,,,,,,,
Large,113424,59003,8930,36,36020,6443,52.02,7.87,31.76,5.68,0.06
Small,92599,46107,7850,46,29822,7089,49.79,8.48,32.21,7.66,0.10


In [58]:
nm = pr_df[~pr_df.is_merged & (pr_df.event_author == "dependabot")]
len(nm), nm["superseded"].eq(True).mean()

(68452, np.float64(0.9611990884123182))

In [59]:
sup = pr_df.groupby("pr_id")["superseded"].apply(lambda s: s.eq(True).any())
nm = pr_df.drop_duplicates("pr_id")
nm = nm[~nm["is_merged"] & (nm["event_author"] == "dependabot")]
len(nm), nm["pr_id"].map(sup).mean()

(68452, np.float64(0.9611990884123182))

In [60]:
pd.set_option('display.max_rows', None)
open_per_repo = (pr_df[pr_df.outcome == "open"]
                 .groupby(["repo_x", "package_ecosystem"]).size())
print(open_per_repo.value_counts().sort_index().head(15))
pd.set_option('display.max_rows', 10)

1     1898
2      829
3      510
4      333
5      352
6      163
7      108
8       84
9       60
10      76
11      34
12      21
13       8
14      11
15       6
Name: count, dtype: int64


In [61]:
repo_stats = pr_df.groupby("repo_x").agg(
    prs=("pr_id", "size"),
    merge_rate=("outcome", lambda s: (s == "merged").mean()),
)
print(repo_stats["merge_rate"].describe())
print("repos that never merged:", (repo_stats.merge_rate == 0).mean() * 100, "%")
print("repos that merged everything:", (repo_stats.merge_rate == 1).mean() * 100, "%")
print("mean of per-repo merge rates:", repo_stats.merge_rate.mean() * 100, "%")

count    10259.000000
mean         0.418525
std          0.417503
min          0.000000
25%          0.000000
50%          0.329004
75%          0.892536
max          1.000000
Name: merge_rate, dtype: float64
repos that never merged: 39.40929915196413 %
repos that merged everything: 21.32761477726874 %
mean of per-repo merge rates: 41.85252531631241 %


In [62]:
r5 = repo_stats[repo_stats.prs >= 5]
print(len(r5), (r5.merge_rate == 0).mean(), (r5.merge_rate == 1).mean(), r5.merge_rate.mean())

5401 0.2521755230512868 0.11720051842251435 0.44790790079623966


In [63]:
from packaging.version import parse as V, InvalidVersion

def bump_type(a, b):
    try:
        va = V(re.search(r"\d+(?:\.\d+)*", str(a)).group())
        vb = V(re.search(r"\d+(?:\.\d+)*", str(b)).group())
    except (AttributeError, InvalidVersion):
        return None
    if vb.major != va.major: return "major"
    if vb.minor != va.minor: return "minor"
    return "patch"

single = pr_df[pr_df.package_version_from.apply(lambda v: isinstance(v, str))].copy()
single["bump_type"] = [bump_type(a, b) for a, b in single[["package_version_from", "package_version_to"]].itertuples(index=False)]
print(pd.crosstab(single.bump_type, single.outcome, normalize="index").mul(100).round(1))
print(single[single.outcome == "merged"].groupby("bump_type")["reverted_ref"].agg(["sum", "count", "mean"]))

outcome    closed_bot  closed_human  closed_superseded  merged  open
bump_type                                                           
major             4.2          13.5               49.3    27.0   6.0
minor             1.3           6.5               33.8    53.8   4.6
patch             1.3           5.0               18.1    68.8   6.9
           sum  count      mean
bump_type                      
major       24  13973  0.001718
minor       37  37544  0.000986
patch       13  44286  0.000294


In [64]:
m = pr_df[pr_df.outcome == "merged"]
print("median ttm by security:", m.groupby(m.is_security_bump)["ttm_h"].median())
print("median ttm by size:", m.groupby("repository_size")["ttm_h"].median())
print("auto-merge by size:", m.groupby("repository_size")["auto_merge"].mean())

median ttm by security: is_security_bump
False     5.493889
True     11.562778
Name: ttm_h, dtype: float64
median ttm by size: repository_size
Large    7.134444
Small    5.101944
Name: ttm_h, dtype: float64
auto-merge by size: repository_size
Large    0.082911
Small    0.100245
Name: auto_merge, dtype: float64


In [65]:
sev = {"LOW": 0, "MODERATE": 1, "HIGH": 2, "CRITICAL": 3}
hits = merged[merged.bump_match].assign(s=lambda x: x.severity.map(sev))
pr_df["max_severity"] = pr_df.pr_id.map(hits.groupby("pr_id")["s"].max().map({v: k for k, v in sev.items()}))
print(pd.crosstab(pr_df.max_severity, pr_df.outcome, normalize="index").mul(100).round(1))
print(pd.crosstab(single.is_security_bump, single.bump_type, normalize="index").mul(100).round(1))

outcome       closed_bot  closed_human  closed_superseded  merged  open
max_severity                                                           
CRITICAL             3.8          12.4               21.0    39.1  23.6
HIGH                 3.2          11.7               30.6    38.0  16.5
LOW                  2.9          11.1               20.4    50.4  15.2
MODERATE             2.9          11.0               28.2    42.7  15.2
bump_type         major  minor  patch
is_security_bump                     
False              29.8   39.4   30.9
True               21.1   31.8   47.1


In [66]:
t = pr_df["pr_title"].str.lower()
pr_df["dep_kind"] = np.select([t.str.contains(r"\(deps-dev\)"), t.str.contains(r"\(deps\)")],
                              ["dev", "prod"], default="unknown")
print(pd.crosstab(pr_df.dep_kind, pr_df.outcome, normalize="index").mul(100).round(1))

outcome   closed_bot  closed_human  closed_superseded  merged  open
dep_kind                                                           
dev              1.9           6.5               35.1    53.2   3.2
prod             2.5           8.3               28.2    54.6   6.5
unknown          2.4           8.8               32.1    48.7   8.0


In [67]:
pr_df["is_group"] = pr_df["pr_title"].str.contains(r"bump the .+ group", case=False, regex=True)
print(pd.crosstab(pr_df.is_group, pr_df.outcome, normalize="index").mul(100).round(1))

outcome   closed_bot  closed_human  closed_superseded  merged  open
is_group                                                           
False            2.3           8.3               32.0    50.6   6.8
True             3.2           6.2               30.6    58.2   1.8


In [68]:
top = pr_df.package_ecosystem.value_counts().head(6).index
sub = pr_df[pr_df.package_ecosystem.isin(top)]
print(pd.crosstab(sub.package_ecosystem, sub.outcome, normalize="index").mul(100).round(1))

outcome            closed_bot  closed_human  closed_superseded  merged  open
package_ecosystem                                                           
ACTIONS                   1.3           4.1               19.8    68.4   6.4
COMPOSER                  1.9           5.7               29.0    60.3   3.2
MAVEN                     1.6           8.7               27.7    57.5   4.4
NPM                       2.4           8.4               33.0    49.4   6.8
PIP                       1.8          12.0               37.2    44.2   4.8
RUBYGEMS                  0.8           6.3               30.5    58.3   4.0


In [69]:
sec = pr_df[pr_df.is_security_bump == True]
pr_df["still_active"] = pr_df["last_commit"] > pd.to_datetime(pr_df["created_at_x"])
print(pd.crosstab([sec.still_active], sec.outcome, normalize="index").mul(100).round(1))
print("open security PRs in inactive repos:",
      (sec[sec.outcome == "open"].still_active == False).mean() * 100, "%")

outcome       closed_bot  closed_human  closed_superseded  merged  open
still_active                                                           
False                2.7           3.7               27.5     0.1  66.1
True                 3.2          12.6               27.0    48.0   9.1
open security PRs in inactive repos: 53.050731261425966 %


In [70]:
rev = result_df[result_df.source_head_ref.str.lower().str.startswith("revert", na=False) & result_df.source_merged]
rev[["repo", "source_number", "source_title"]].to_csv("reverts_to_label.csv", index=False)

In [71]:
mr = lambda s: round((s == "merged").mean() * 100, 1)
dec = pr_df[pr_df.outcome.isin(["merged", "closed_human"])]
print("decided merge rate:", mr(dec.outcome))
print("decided by security:", dec.groupby("is_security_bump")["outcome"].apply(mr).to_dict())
print("decided by size:", dec.groupby("repository_size")["outcome"].apply(mr).to_dict())
d1 = single[single.outcome.isin(["merged", "closed_human"])]
print("decided by update type:", d1.groupby("bump_type")["outcome"].apply(mr).to_dict())
print("update type n:", single.bump_type.value_counts().to_dict())

print("security unknown:", pr_df.is_security_bump.isna().sum(),
      " both methods:", ((pr_df.is_security_bump == True) & pr_df.security_in_body).sum())
print(pd.crosstab(pr_df.is_security_bump, pr_df.outcome, normalize="index").mul(100).round(1))
mg = pr_df[pr_df.outcome == "merged"]
print(mg.groupby("is_security_bump")["reverted_ref"].agg(["sum", "count"]))
for lab, s in [("security", pr_df[pr_df.is_security_bump == True]), ("version", pr_df[pr_df.is_security_bump == False])]:
    o = s[s.outcome == "open"]
    print(lab, "open in inactive %:", round((~o.still_active).mean() * 100, 1),
          " PRs in inactive %:", round((~s.still_active).mean() * 100, 1))
print("severity n:", pr_df.max_severity.value_counts().to_dict())

print("reverted repos:", pr_df[pr_df.reverted_ref].repo_x.nunique())
print("repos with open PRs:", pr_df[pr_df.outcome == "open"].repo_x.nunique())
print("grouped n:", pr_df.is_group.sum())
print("ecosystem n:", pr_df.package_ecosystem.value_counts().head(6).to_dict(),
      " non-npm %:", round((pr_df.package_ecosystem != "NPM").mean() * 100, 1))

print(pd.crosstab(pr_df.repository_size, pr_df.outcome, normalize="index").mul(100).round(1))
pr_df["bucket"] = pd.cut(pr_df.stars, [0, 24, 99, 499, 4999, 10**9],
                         labels=["<=24", "25-99", "100-499", "500-4999", "5000+"])
print(pd.crosstab(pr_df.bucket, pr_df.outcome, normalize="index").mul(100).round(1))

decided merge rate: 86.2
decided by security: {False: 88.5, True: 78.4}
decided by size: {'Large': 86.9, 'Small': 85.5}
decided by update type: {'major': 66.6, 'minor': 89.3, 'patch': 93.3}
update type n: {'minor': 69785, 'patch': 64379, 'major': 51835}
security unknown: 1188  both methods: 12142
outcome           closed_bot  closed_human  closed_superseded  merged  open
is_security_bump                                                           
False                    2.0           7.0               33.6    54.2   3.1
True                     3.1          11.4               27.1    41.6  16.8
                  sum  count
is_security_bump            
False              61  82873
True               18  21601
security open in inactive %: 53.1  PRs in inactive %: 13.5
version open in inactive %: 38.2  PRs in inactive %: 4.4
severity n: {'MODERATE': 22509, 'HIGH': 16330, 'CRITICAL': 7395, 'LOW': 5749}
reverted repos: 68
repos with open PRs: 4127
grouped n: 11178
ecosystem n: {'NPM': 17611

In [72]:
gb = pr_df[pr_df.is_group & (pr_df.outcome == "closed_bot")]
print(len(gb))
print(gb["event_author"].value_counts())

353
event_author
dependabot        332
github-actions     18
stale               3
Name: count, dtype: int64


In [74]:
#### dc = comments_df[comments_df.pr_id.isin(gb.pr_id) & comments_df.comment_author.eq("dependabot")]
print(dc.comment_body.fillna("").str.split("\n").str[0].str[:100].value_counts().head(15))

NameError: name 'dc' is not defined

In [75]:
print(gb[["repo_x", "pr_number", "event_author"]].sample(10, random_state=1).to_string())

                               repo_x  pr_number event_author
82338            WFCD/warframe-status       2047   dependabot
96160   dataaxiom/ghcr-cleanup-action         76   dependabot
54357             WordPress/gutenberg      68989   dependabot
176658    BeaconCMS/beacon_live_admin        404   dependabot
176483      adamlui/chatgpt-auto-talk         88   dependabot
185521                  zazuko/trifid        657   dependabot
192076              team-apm/apm-data        909   dependabot
169964            glen-cheney/Shuffle        650   dependabot
126745        hifiwi-fi/breadcrum.net        564   dependabot
98386      adamlui/chatgpt-widescreen        242   dependabot


In [76]:
cols = ["merged", "closed_superseded", "closed_human", "closed_bot", "open"]
def tab(df, key):
    t = pd.crosstab(df[key], df.outcome, normalize="index").mul(100).round(1).reindex(columns=cols)
    t.insert(0, "n", df[key].value_counts())
    print(t, "\n")

o = pr_df.outcome
print("[A] Table 4.1:\n", pd.DataFrame({"n": o.value_counts(), "%": o.value_counts(normalize=True).mul(100).round(1)}), "\n")
print("[B] no decision %:", round(o.isin(["closed_superseded", "open"]).mean() * 100, 1))
nm = pr_df[~pr_df.is_merged & (pr_df.event_author == "dependabot")]
print("[C] Dependabot closures superseded %:", round(nm.superseded.eq(True).mean() * 100, 1), "\n")
print("[D] Table 4.2:"); tab(single, "bump_type")
print("[E] grouped row:"); tab(pr_df, "is_group")
print("[F] Table 4.4:"); tab(pr_df, "is_security_bump")
print("[G] Table 4.5:"); tab(pr_df[pr_df.is_security_bump == True], "max_severity")
print("[H] Table 4.6:"); tab(pr_df, "repository_size")
print("[I] Table 4.7:"); tab(pr_df, "bucket")

[A] Table 4.1:
                         n     %
outcome                        
merged             105110  51.0
closed_superseded   65842  32.0
closed_human        16780   8.1
open                13532   6.6
closed_bot           4759   2.3 

[B] no decision %: 38.5
[C] Dependabot closures superseded %: 96.1 

[D] Table 4.2:
outcome        n  merged  closed_superseded  closed_human  closed_bot  open
bump_type                                                                  
major      51835    27.0               49.3          13.5         4.2   6.0
minor      69785    53.8               33.8           6.5         1.3   4.6
patch      64379    68.8               18.1           5.0         1.3   6.9 

[E] grouped row:
outcome        n  merged  closed_superseded  closed_human  closed_bot  open
is_group                                                                   
False     194845    50.6               32.0           8.3         2.3   6.8
True       11178    58.2               30.6    

In [77]:
typed = single.loc[single.bump_type.notna(), "pr_id"]
rest = pr_df[~pr_df.pr_id.isin(typed)]
print("not in table:", len(rest))
print("grouped:", rest.is_group.sum())
print("other multi-package:", (~rest.is_group & ~rest.pr_id.isin(single.pr_id)).sum())
print("version not SemVer:", rest.pr_id.isin(single.pr_id).sum())

not in table: 20024
grouped: 11178
other multi-package: 8836
version not SemVer: 10


In [78]:
unk = pr_df[pr_df["is_security_bump"].isna()]
print(len(unk), unk["package_ecosystem"].str.upper().isin(unsupported_ecosystems).sum())

1188 1153


In [79]:
multi = pr_df[~pr_df.is_group & ~pr_df.pr_id.isin(single.pr_id)]
print(len(multi))
print(multi.outcome.value_counts(normalize=True).mul(100).round(1).reindex(cols))

8836
outcome
merged               31.7
closed_superseded    18.3
closed_human         15.5
closed_bot            5.3
open                 29.2
Name: proportion, dtype: float64


In [80]:
multi = pr_df[~pr_df.is_group & ~pr_df.pr_id.isin(single.pr_id)]
kind = multi.package_version_from.apply(lambda v: "list" if isinstance(v, list) else ("missing" if v is None or (isinstance(v, float) and pd.isna(v)) else type(v).__name__))
print(len(multi))
print(kind.value_counts())
print(multi.loc[kind == "missing", "pr_title"].head(15).to_string())

8836
package_version_from
list       8586
NAType      210
missing      40
Name: count, dtype: int64
26985                        Bump System.Text.Json to 8.0.5
44572       feat!: Upgrade action to use Node.js 24 and ESM
53756     ⬆️👷 Update actions/checkout requirement to 8ed...
58421     ⬆️👷 Update actions/checkout requirement to c2d...
67528             Bump brace-expansion in /Season-2/Level-3
78162                     Update System.Text.Json to 6.0.10
78163         Update Microsoft.Rest.ClientRuntime to 2.3.24
82746     ⬆️👷 Update actions/checkout requirement to 8e8...
99236           Bump SixLabors.ImageSharp to 2.1.11, 3.1.11
100721    Update grafana/plugin-actions requirement to b...
102281    Update Microsoft.AspNetCore.Components.WebAsse...
114879    ⬆️👷 Update actions/checkout requirement to ff7...
116087    ⬆️👷 Update actions/checkout requirement to 09d...
129529    ⬆️👷 Update actions/checkout requirement to 85e...
131440                          npm(deps): bump cross-spawn


In [81]:
ml = multi[kind == "list"]
print(len(ml))
print(ml.outcome.value_counts(normalize=True).mul(100).round(1).reindex(["merged", "closed_superseded", "closed_human", "closed_bot", "open"]))

8586
outcome
merged               31.7
closed_superseded    18.4
closed_human         15.7
closed_bot            5.3
open                 29.0
Name: proportion, dtype: float64


In [82]:
print("security %:", round((ml.is_security_bump == True).mean() * 100, 1))
print("inactive %:", round((~ml.still_active.fillna(False).astype(bool)).mean() * 100, 1))

security %: 79.6
inactive %: 20.6


In [83]:
for name, d in [("grouped", pr_df[pr_df.is_group]), ("multi-package", ml)]:
    m = d[d.outcome == "merged"]
    print(name, len(m), m.reverted_ref.sum(), round(m.reverted_ref.mean() * 100, 3))

grouped 6502 0 0.0
multi-package 2725 7 0.257


In [84]:
both = pd.concat([pr_df[pr_df.is_group], ml])
print(len(both))
print(both.outcome.value_counts(normalize=True).mul(100).round(1).reindex(["merged", "closed_superseded", "closed_human", "closed_bot", "open"]))

19764
outcome
merged               46.7
closed_superseded    25.3
closed_human         10.3
closed_bot            4.1
open                 13.6
Name: proportion, dtype: float64


In [85]:
m = both[both.outcome == "merged"]
print(len(m), m.reverted_ref.sum(), round(m.reverted_ref.mean() * 100, 3))

9227 7 0.076


In [86]:
typed = single.loc[single.bump_type.notna(), "pr_id"]
rev = pr_df[(pr_df.outcome == "merged") & pr_df.reverted_ref
            & ~pr_df.pr_id.isin(typed) & ~pr_df.pr_id.isin(both.pr_id)]
print(rev[["repo_x", "pr_number", "pr_title", "package_name", "package_version_from", "package_version_to"]].to_string())

                                repo_x  pr_number                pr_title package_name package_version_from package_version_to
168155  dxc-technology/halstack-client        112  Bump form-data in /lib          NaN                  NaN                NaN


In [91]:
bots = { "dependabot", "github-actions", "mergify", "accept-to-ship-bot", "stale", "kodiakhq", "tekton-robot", "FreeTubeBot" }

merged = pr_df[pr_df.outcome == "merged"]
mergers = (events_df[events_df.event_type == "MergedEvent"]
           .sort_values("created_at").groupby("pr_id", as_index=False).last()[["pr_id", "event_author"]]
           .rename(columns={"event_author": "merger_login"}))
m = merged.merge(mergers, on="pr_id", how="left")

m["merger"] = np.where(m.merger_login.isna(), "unknown",
               np.where(m.merger_login.isin(bots) | m.merger_login.str.contains(r"\[bot\]|bot$", case=False, na=False), "bot", "human"))
print(len(m))
print(m.merger.value_counts())
print(m.merger.value_counts(normalize=True).mul(100).round(1))
print(m[m.merger == "bot"].merger_login.value_counts().head(15))

auto = events_df[events_df.event_type.isin(["AutoMergeEnabledEvent", "AutoSquashEnabledEvent", "AutoRebaseEnabledEvent"])].pr_id.unique()
m["auto_enabled"] = m.pr_id.isin(auto)
print(pd.crosstab(m.merger, m.auto_enabled, margins=True))

105110
merger
human      78927
bot        25945
unknown      238
Name: count, dtype: int64
merger
human      75.1
bot        24.7
unknown     0.2
Name: proportion, dtype: float64
merger_login
github-actions          15301
dependabot               7543
mergify                  1339
tekton-robot              506
FreeTubeBot               343
                        ...  
asyncapi-bot               68
nodejs-github-bot          63
otto-the-bot               47
companion-module-bot       39
greenbonebot               23
Name: count, Length: 15, dtype: int64
auto_enabled  False  True     All
merger                           
bot           19173  6772   25945
human         75950  2977   78927
unknown         233     5     238
All           95356  9754  105110


In [90]:
events_df

,pr_id,repo,event_nr,event_type,event_author,created_at,data
0,PR_kwDODjJbPM6NAIQJ,UK-Export-Finance/dtfs2,0,PullRequestCommit,dependabot[bot],2025-03-01T00:55:53Z,"{""oid"": ""4f4bb12fa549439933dde43c8f8b037e4cb66..."
1,PR_kwDODjJbPM6NAIQJ,UK-Export-Finance/dtfs2,1,LabeledEvent,dependabot,2025-03-01T00:55:55Z,"{""label"": ""chore""}"
2,PR_kwDODjJbPM6NAIQJ,UK-Export-Finance/dtfs2,2,DeployedEvent,NaN,NaN,{}
3,PR_kwDODjJbPM6NAIQJ,UK-Export-Finance/dtfs2,3,DeployedEvent,NaN,NaN,{}
4,PR_kwDODjJbPM6NAIQJ,UK-Export-Finance/dtfs2,4,DeployedEvent,NaN,NaN,{}
...,...,...,...,...,...,...,...
1921769,PR_kwDOOspSC86eM6fE,SocketDev/socket-mcp,5,IssueComment,NaN,NaN,{}
1921770,PR_kwDOOspSC86eM6fE,SocketDev/socket-mcp,6,IssueComment,NaN,NaN,{}
1921771,PR_kwDOOspSC86eM6fE,SocketDev/socket-mcp,7,MergedEvent,bcomnes,2025-07-09T23:58:14Z,"{""merge_commit_oid"": ""053942a8b56affa4f1f500a5..."
1921772,PR_kwDOOspSC86eM6fE,SocketDev/socket-mcp,8,ClosedEvent,bcomnes,2025-07-09T23:58:14Z,{}
